# The cut that doesn't travel: what Combine route drills miss about NFL route breaks

**NFL Big Data Bowl 2027 · code notebook for the writeup.**

Do the ways a wide receiver breaks in the Combine route drills carry into how they break in regular-season games,
and into how open they get? This notebook reproduces every number and figure in the writeup from the competition
data. It writes the project's code (identical to the repository), then runs six steps:

| Step | What it does |
|---|---|
| `00_audit` | Data checks: regular-season filter, counts, what `a` and `dis` measure |
| `01_combine_features` | One cut detector on the WR position drills; combine cut features per player |
| `02_game_features` | The same detector on 30,499 regular-season routes; game features and separation over expected |
| `03_models` | The pre-registered tests (combine → game, game break → separation, combine → separation) |
| `03b_sensitivity` | Checks added after them: matched combine cuts, a positive control, play context, man vs zone, what a yard of separation is worth |
| `04_writeup` | The writeup's figures and the numbers they cite |

Every step ends with hard checks (`assert`), so a finished run means every check passed. Runtime is about
15–20 minutes on a Kaggle CPU session.

## Setup

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Find the competition CSVs: Kaggle's input folder, or BDB_RAW_DIR when run elsewhere.
candidates = [Path(os.environ["BDB_RAW_DIR"])] if "BDB_RAW_DIR" in os.environ else []
candidates += sorted(p.parent for p in Path("/kaggle/input").rglob("players.csv")) if Path("/kaggle/input").exists() else []
RAW = next((c for c in candidates if (c / "players.csv").exists()), None)
assert RAW is not None, "Attach the NFL Big Data Bowl 2027 competition data to this notebook."
os.environ["BDB_RAW_DIR"] = str(RAW)
print("competition data:", RAW)

try:
    import duckdb  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb>=1.1"], check=True)

PROJECT = Path("bdb").resolve()
for d in ["src", "notebooks", "reports/figures", "logs"]:
    (PROJECT / d).mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT)
print("project folder:", PROJECT)

## Project code

The cells below write the project's modules and step scripts exactly as they are in the repository. They are long;
the results follow in the next sections.

In [ ]:
Path("src/__init__.py").write_text('')

In [ ]:
%%writefile src/cuts.py
"""Cut detection and per-cut mechanics from 10 Hz x/y tracking.

Works on any single-player trajectory (a combine rep or an in-game route), and
derives everything from smoothed x/y so combine and game data get identical
processing. The provided `a` is an unsigned magnitude in both sources, so it is
not used.

A cut is a local peak in the signed turn angle between the velocity vectors
HALF_WINDOW before and after a frame. Using velocities on either side, rather
than frame-to-frame heading, keeps plant-and-reverse breaks measurable: the
heading is undefined at the near-zero apex speed of a shuttle turn or a curl.
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from scipy.signal import find_peaks, savgol_filter

HZ = 10
DT = 1 / HZ

# Smoothing: Savitzky-Golay over 0.7 s, quadratic.
SG_WINDOW = 7
SG_POLY = 2

# Detection.
HALF_WINDOW = 5      # frames (0.5 s) either side for the turn angle
MIN_TURN = 30.0      # degrees
MIN_SEP = 5          # frames between peaks of the same direction
APEX_SEARCH = 3      # frames either side of the turn peak to find the speed minimum
MERGE_WITHIN = 4     # frames: apexes closer than this are one cut
MIN_MOVING = 1.0     # yd/s: both velocity vectors must be at least this fast
MIN_ENTRY = 2.0      # yd/s: peak speed into the cut
MIN_EXIT = 1.5       # yd/s: peak speed out of the cut

# Metric windows (frames).
ENTRY_WINDOW = 15    # 1.5 s before the apex
DECEL_WINDOW = 10    # 1.0 s before the apex
EXIT_WINDOW = 10     # 1.0 s after the apex
LOAD_WINDOW = 5      # 0.5 s either side of the apex
GAIN_FRAMES = 5      # speed gained 0.5 s after the apex


def kinematics(x: np.ndarray, y: np.ndarray) -> dict[str, np.ndarray]:
    """Smoothed velocity and acceleration, split into along-track and lateral parts.

    Signs: `a_long` > 0 speeding up, < 0 slowing down. `a_lat` and `turn_rate`
    > 0 turning left (counter-clockwise viewed from above), < 0 turning right.
    """
    n = len(x)
    w = min(SG_WINDOW, n if n % 2 else n - 1)
    if w <= SG_POLY:
        raise ValueError(f"trajectory too short ({n} frames)")
    vx = savgol_filter(x, w, SG_POLY, deriv=1, delta=DT)
    vy = savgol_filter(y, w, SG_POLY, deriv=1, delta=DT)
    ax = savgol_filter(x, w, SG_POLY, deriv=2, delta=DT)
    ay = savgol_filter(y, w, SG_POLY, deriv=2, delta=DT)
    speed = np.hypot(vx, vy)
    safe = np.where(speed > 1e-6, speed, np.nan)
    a_long = (vx * ax + vy * ay) / safe
    a_lat = (vx * ay - vy * ax) / safe
    return {
        "vx": vx, "vy": vy, "speed": speed,
        "a_long": np.nan_to_num(a_long),
        "a_lat": np.nan_to_num(a_lat),
        "turn_rate": np.where(speed > MIN_MOVING, a_lat / safe, 0.0),  # rad/s
    }


def turn_angle(vx: np.ndarray, vy: np.ndarray, half: int = HALF_WINDOW) -> np.ndarray:
    """Signed angle (deg) from the velocity `half` frames before to `half` frames after.

    > 0 is a left turn. 0 where either vector is slower than MIN_MOVING or out of range.
    """
    n = len(vx)
    theta = np.zeros(n)
    if n <= 2 * half:
        return theta
    i = np.arange(half, n - half)
    x1, y1, x2, y2 = vx[i - half], vy[i - half], vx[i + half], vy[i + half]
    ang = np.degrees(np.arctan2(x1 * y2 - y1 * x2, x1 * x2 + y1 * y2))
    moving = (np.hypot(x1, y1) >= MIN_MOVING) & (np.hypot(x2, y2) >= MIN_MOVING)
    theta[i] = np.where(moving, ang, 0.0)
    return theta


def detect_cuts(k: dict[str, np.ndarray]) -> list[dict]:
    """Cuts in one trajectory, as dicts of apex index, signed turn and metrics."""
    speed, a_long, a_lat = k["speed"], k["a_long"], k["a_lat"]
    n = len(speed)
    theta = turn_angle(k["vx"], k["vy"])

    cands = []
    for sign in (1, -1):
        peaks, _ = find_peaks(sign * theta, height=MIN_TURN, distance=MIN_SEP)
        for p in peaks:
            lo, hi = max(p - APEX_SEARCH, 0), min(p + APEX_SEARCH + 1, n)
            apex = lo + int(np.argmin(speed[lo:hi]))
            cands.append((apex, float(theta[p])))

    # Merge candidates whose apexes coincide (keep the larger turn).
    cands.sort(key=lambda c: -abs(c[1]))
    kept: list[tuple[int, float]] = []
    for apex, th in cands:
        if all(abs(apex - a) >= MERGE_WITHIN for a, _ in kept):
            kept.append((apex, th))
    kept.sort()

    cuts = []
    for apex, th in kept:
        e0 = max(apex - ENTRY_WINDOW, 0)
        entry_i = e0 + int(np.argmax(speed[e0:apex + 1]))
        entry_speed = float(speed[entry_i])
        x1 = min(apex + EXIT_WINDOW + 1, n)
        exit_speed = float(speed[apex:x1].max())
        if entry_speed < MIN_ENTRY or exit_speed < MIN_EXIT:
            continue
        apex_speed = float(speed[apex])
        d0 = max(apex - DECEL_WINDOW, 0)
        l0, l1 = max(apex - LOAD_WINDOW, 0), min(apex + LOAD_WINDOW + 1, n)
        gain_i = apex + GAIN_FRAMES
        # Distance travelled while slowing from entry speed to the apex.
        decel_dist = float(np.sum(speed[entry_i:apex]) * DT)
        cuts.append({
            "apex_frame": apex,
            "turn_deg": abs(th),
            "turn_dir": "L" if th > 0 else "R",
            "entry_speed": entry_speed,
            "apex_speed": apex_speed,
            "exit_speed": exit_speed,
            "speed_retention": apex_speed / entry_speed,
            "peak_decel": float(-a_long[d0:apex + 1].min()),
            "decel_time": (apex - entry_i) * DT,
            "decel_dist": decel_dist,
            "peak_lat_accel": float(np.abs(a_lat[l0:l1]).max()),
            "reaccel_gain": float(speed[gain_i] - apex_speed) if gain_i < n else np.nan,
            "peak_reaccel": float(a_long[apex:x1].max()) if x1 - apex >= 3 else np.nan,
            "frames_after_apex": n - 1 - apex,
        })
    return cuts


def cuts_for_reps(frames: pd.DataFrame, rep_cols: list[str]) -> pd.DataFrame:
    """Detect cuts in every rep. `frames` needs rep_cols + time, x, y."""
    out = []
    for key, r in frames.sort_values([*rep_cols, "time"]).groupby(rep_cols, sort=False):
        if len(r) <= 2 * HALF_WINDOW:
            continue
        k = kinematics(r["x"].to_numpy(float), r["y"].to_numpy(float))
        key = key if isinstance(key, tuple) else (key,)
        for i, c in enumerate(detect_cuts(k)):
            out.append({**dict(zip(rep_cols, key)), "cut_idx": i, "n_frames": len(r), **c})
    return pd.DataFrame(out)

In [ ]:
%%writefile src/data.py
"""Data access: CSV -> Parquet conversion and a DuckDB connection with the
standard views every phase uses.

Named `data` rather than `io` so it never shadows the stdlib module.
"""
from __future__ import annotations

import os
from pathlib import Path

import duckdb

ROOT = Path(__file__).resolve().parents[1]
# The competition CSVs: data/raw/ by default, or BDB_RAW_DIR (e.g. Kaggle's read-only input folder).
RAW = Path(os.environ.get("BDB_RAW_DIR", ROOT / "data" / "raw"))
PARQUET = ROOT / "data" / "parquet"

TABLES = [
    "players",
    "combine_results",
    "combine_tracking",
    "games",
    "player_play",
    "player_career_successes",
]
GAME_TRACKING_SEASONS = [2023, 2024, 2025]


def to_parquet(overwrite: bool = False) -> None:
    """Convert every CSV in data/raw/ to data/parquet/ (NA -> NULL)."""
    PARQUET.mkdir(parents=True, exist_ok=True)
    con = duckdb.connect()
    for csv in sorted(RAW.glob("*.csv")):
        out = PARQUET / f"{csv.stem}.parquet"
        if out.exists():
            if not overwrite:
                continue
            out.unlink()
        # sample_size=-1: infer types from the whole file (many columns are
        # NA for thousands of rows before the first real value).
        con.execute(
            f"COPY (SELECT * FROM read_csv('{csv}', nullstr='NA', sample_size=-1)) "
            f"TO '{out}' (FORMAT parquet)"
        )


def connect() -> duckdb.DuckDBPyConnection:
    """In-memory DuckDB connection with one view per table, plus:

    game_tracking      all tracking frames, tagged with season / season_type / week
    game_tracking_reg  regular-season frames only (the competition's outcome window)
    player_play_reg    regular-season player-play rows only
    """
    if not (PARQUET / "players.parquet").exists():
        to_parquet()
    con = duckdb.connect()
    for t in TABLES:
        con.execute(f"CREATE VIEW {t} AS SELECT * FROM '{PARQUET / t}.parquet'")
    files = ", ".join(f"'{PARQUET}/game_tracking_{y}.parquet'" for y in GAME_TRACKING_SEASONS)
    con.execute(
        f"""CREATE VIEW game_tracking AS
        SELECT t.*, g.season, g.season_type, g.week
        FROM read_parquet([{files}]) t JOIN games g USING (game_id)"""
    )
    con.execute("CREATE VIEW game_tracking_reg AS SELECT * FROM game_tracking WHERE season_type = 'REG'")
    con.execute(
        """CREATE VIEW player_play_reg AS
        SELECT pp.*, g.season, g.week
        FROM player_play pp JOIN games g USING (game_id)
        WHERE g.season_type = 'REG'"""
    )
    return con

In [ ]:
%%writefile src/features.py
"""Player-level combine cut features.

Raw cut metrics are not comparable across breaks (a 180-degree curl costs far
more speed than a 45-degree slant break), and players ran different drill sets.
So each metric is z-scored within its *slot*: the same break of the same drill
across players. A player's feature is the mean of those z-scores over their
cuts, so 0 = average for the breaks they ran.

Slots in route drills are drill x turn direction x order (the curl's plant is
CURL_ROUTE_RIGHT|L1, the turn upfield after the catch is CURL_ROUTE_RIGHT|R1),
because each route breaks a fixed way. In the shuttle and 3-cone the player
chooses which way to turn, so slots there are drill x cut order.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from src.cuts import cuts_for_reps

# metric -> description
METRICS = {
    "entry_speed": "speed into the break (yd/s)",
    "speed_retention": "apex speed / entry speed",
    "peak_decel": "peak braking (yd/s^2)",
    "decel_dist": "distance used to slow down (yd)",
    "peak_lat_accel": "peak lateral (turning) acceleration (yd/s^2)",
    "reaccel_gain": "speed regained 0.5 s after the apex (yd/s)",
}
POSITION_DRILLS = ("SKILL_DRILLS_WR",)
AGILITY_DRILLS = ("SHORT_SHUTTLE", "THREE_CONE_DRILL")
MIN_CUTS_PER_SLOT = 20


def combine_cuts(con, drill_types: tuple[str, ...]) -> pd.DataFrame:
    """Detect cuts in every combine rep of the given drill types."""
    types = ", ".join(f"'{t}'" for t in drill_types)
    frames = con.sql(
        f"""SELECT event_id, nfl_id, drill_type, drill_name, time, x, y
            FROM combine_tracking WHERE entity_type = 'PLAYER' AND drill_type IN ({types})"""
    ).df()
    cuts = cuts_for_reps(frames, ["event_id"])
    reps = frames.drop_duplicates("event_id").set_index("event_id")[["nfl_id", "drill_type", "drill_name"]]
    return cuts.join(reps, on="event_id")


def slot_z(cuts: pd.DataFrame, by_direction: bool, metrics=tuple(METRICS)) -> pd.DataFrame:
    """Add `slot` and `<metric>_z` (z-score within slot). Slots with < MIN_CUTS_PER_SLOT cuts are dropped."""
    c = cuts.sort_values(["event_id", "apex_frame"]).copy()
    if by_direction:
        order = c.groupby(["event_id", "turn_dir"]).cumcount() + 1
        c["slot"] = c["drill_name"] + "|" + c["turn_dir"] + order.astype(str)
    else:
        c["slot"] = c["drill_name"] + "|" + (c["cut_idx"] + 1).astype(str)
    c = c[c.groupby("slot")["event_id"].transform("size") >= MIN_CUTS_PER_SLOT].copy()
    for m in metrics:
        g = c.groupby("slot")[m]
        c[f"{m}_z"] = (c[m] - g.transform("mean")) / g.transform("std")
    return c


def player_features(zcuts: pd.DataFrame, prefix: str, metrics=tuple(METRICS)) -> pd.DataFrame:
    """One row per player: mean drill-z per metric, plus left/right asymmetry and consistency.

    `<prefix>_<metric>`          mean z over all cuts
    `<prefix>_<metric>_lr`       mean z on left cuts minus mean z on right cuts
    `<prefix>_<metric>_sd`       SD of z across cuts (lower = more consistent)
    """
    g = zcuts.groupby("nfl_id")
    feats = pd.DataFrame({
        f"{prefix}_n_cuts": g.size(),
        f"{prefix}_n_drills": g["drill_name"].nunique(),
    })
    for m in metrics:
        z = f"{m}_z"
        feats[f"{prefix}_{m}"] = g[z].mean()
        side = zcuts.pivot_table(index="nfl_id", columns="turn_dir", values=z, aggfunc="mean")
        feats[f"{prefix}_{m}_lr"] = side.get("L") - side.get("R")
        feats[f"{prefix}_{m}_sd"] = g[z].std()
    return feats


def split_half_reliability(zcuts: pd.DataFrame, n_iter: int = 200, seed: int = 0,
                           metrics=tuple(METRICS), min_drills: int = 4) -> pd.DataFrame:
    """Spearman-Brown corrected split-half reliability of each player feature.

    Each iteration splits every player's drills at random into two halves (a
    drill's cuts stay together), computes the feature on each half, and
    correlates the halves across players.
    """
    rng = np.random.default_rng(seed)
    z = zcuts[zcuts.groupby("nfl_id")["drill_name"].transform("nunique") >= min_drills]
    pairs = z[["nfl_id", "drill_name"]].drop_duplicates()
    stats = {k: [] for m in metrics for k in (m, f"{m}_lr", f"{m}_sd")}
    for _ in range(n_iter):
        half = pairs.assign(h=rng.random(len(pairs)))
        half["h"] = half.groupby("nfl_id")["h"].rank(method="first") % 2
        zz = z.merge(half, on=["nfl_id", "drill_name"])
        a = player_features(zz[zz.h == 0], "f", metrics)
        b = player_features(zz[zz.h == 1], "f", metrics)
        for k in stats:
            r = a[f"f_{k}"].corr(b[f"f_{k}"])
            stats[k].append(2 * r / (1 + r) if r > -1 else np.nan)
    players = z["nfl_id"].nunique()
    return pd.DataFrame(
        [{"feature": k, "reliability": np.nanmedian(v), "players": players} for k, v in stats.items()]
    )

In [ ]:
%%writefile src/models.py
"""Regression helpers for Phase 3: OLS with robust or player-clustered errors, ridge, bootstrap, Holm.

numpy / scipy only. Every model adds its own intercept; pass predictors without one.
"""
from __future__ import annotations

import numpy as np
import pandas as pd
from scipy import stats

UDFA_PICK = 260  # UDFAs sit just after the last pick (256 in this cohort); a UDFA indicator goes alongside


def baseline_matrix(df: pd.DataFrame, medians: pd.Series | None = None) -> tuple[pd.DataFrame, pd.Series]:
    """The pre-registered baseline: log(draft pick) with UDFAs at pick 260, a UDFA flag, 40, 10-yd split,
    vertical, broad jump, height, weight. Missing tests are filled with `medians` (default: this sample's)."""
    b = pd.DataFrame({
        "log_pick": np.log(df.draft_overall_pick.fillna(UDFA_PICK).astype(float)),
        "udfa": df.draft_overall_pick.isna().astype(float),
        "forty": df.forty, "ten_yd_split": df.ten_yd_split, "vertical": df.vertical,
        "broad_jump": df.broad_jump, "height": df.combine_height, "weight": df.combine_weight,
    }, index=df.index).astype(float)
    medians = b.median() if medians is None else medians
    return b.fillna(medians), medians


def zscore(X: pd.DataFrame, ref: pd.DataFrame | None = None) -> pd.DataFrame:
    """Standardize with `ref`'s mean and SD (default: X's own). Constant columns become 0."""
    ref = X if ref is None else ref
    sd = ref.std(ddof=0).replace(0, 1.0)
    return (X - ref.mean()) / sd


def ols(y: np.ndarray, X: np.ndarray, cluster: np.ndarray | None = None) -> dict:
    """OLS of y on [1, X]. Standard errors are HC3, or CR1 when `cluster` labels are given.

    p-values use a t distribution with n - k degrees of freedom (HC3) or G - 1 (CR1, G clusters).
    `partial_r` is the partial correlation of each predictor with y, from the classical t:
    t / sqrt(t^2 + n - k).
    """
    y = np.asarray(y, float)
    X1 = np.column_stack([np.ones(len(y)), np.asarray(X, float)])
    n, k = X1.shape
    XtX_inv = np.linalg.pinv(X1.T @ X1)
    beta = XtX_inv @ X1.T @ y
    e = y - X1 @ beta
    if cluster is None:
        h = np.einsum("ij,jk,ik->i", X1, XtX_inv, X1)
        meat = (X1 * (e / (1 - h))[:, None] ** 2).T @ X1
        df = n - k
    else:
        codes, groups = pd.factorize(np.asarray(cluster))
        G = len(groups)
        scores = np.zeros((G, k))
        np.add.at(scores, codes, X1 * e[:, None])
        meat = scores.T @ scores * (G / (G - 1)) * ((n - 1) / (n - k))
        df = G - 1
    se = np.sqrt(np.diag(XtX_inv @ meat @ XtX_inv))
    t = beta / se
    t_classical = beta / np.sqrt(np.diag(XtX_inv) * (e @ e) / (n - k))
    return {
        "coef": beta, "se": se, "t": t, "p": 2 * stats.t.sf(np.abs(t), df),
        "partial_r": t_classical / np.sqrt(t_classical ** 2 + (n - k)), "n": n, "df": df,
    }


def ridge_gcv(X: np.ndarray, y: np.ndarray, lambdas: np.ndarray | None = None) -> tuple[float, np.ndarray, float]:
    """Ridge on standardized-by-caller X with the penalty chosen by generalized cross-validation.

    Returns (intercept, coefficients, lambda). The intercept is not penalized.
    """
    X, y = np.asarray(X, float), np.asarray(y, float)
    lambdas = np.logspace(-3, 3, 61) * len(y) if lambdas is None else lambdas
    xm, ym = X.mean(axis=0), y.mean()
    Xc, yc = X - xm, y - ym
    U, d, Vt = np.linalg.svd(Xc, full_matrices=False)
    uty = U.T @ yc
    best = None
    for lam in lambdas:
        f = d ** 2 / (d ** 2 + lam)
        fit = U @ (f * uty)
        gcv = len(y) * np.sum((yc - fit) ** 2) / (len(y) - f.sum()) ** 2
        if best is None or gcv < best[0]:
            best = (gcv, lam)
    lam = best[1]
    beta = Vt.T @ ((d / (d ** 2 + lam)) * uty)
    return ym - xm @ beta, beta, lam


def leave_one_group_out(df: pd.DataFrame, target: str, build, group: str) -> pd.Series:
    """Out-of-sample ridge predictions of `target`, holding out one value of `group` at a time.

    `build(train, test)` returns (X_train, X_test) as DataFrames; it sees only the training rows when it
    computes anything from data (medians, means, SDs), so nothing from the held-out group leaks in.
    """
    pred = pd.Series(np.nan, index=df.index)
    for g in df[group].unique():
        train, test = df[df[group] != g], df[df[group] == g]
        Xtr, Xte = build(train, test)
        a, b, _ = ridge_gcv(Xtr.to_numpy(), train[target].to_numpy())
        pred[test.index] = a + Xte.to_numpy() @ b
    return pred


def oos_r2(y: pd.Series, pred: pd.Series) -> float:
    """Pooled out-of-sample R^2: 1 - SSE / total sum of squares around the pooled mean."""
    return float(1 - np.sum((y - pred) ** 2) / np.sum((y - y.mean()) ** 2))


def bootstrap(stat, units: np.ndarray, n_boot: int = 2000, seed: int = 0) -> np.ndarray:
    """`stat(index)` over `n_boot` resamples of `units` (players) with replacement.

    `index` is an integer array of positions into `units`; with clusters, the caller expands each drawn
    unit to its rows. Failed resamples (e.g. a singular fit) come back as NaN.
    """
    rng = np.random.default_rng(seed)
    out = np.full(n_boot, np.nan)
    for i in range(n_boot):
        try:
            out[i] = stat(rng.integers(0, len(units), len(units)))
        except (np.linalg.LinAlgError, ValueError, ZeroDivisionError):
            pass
    return out


def percentile_ci(draws: np.ndarray, level: float = 0.95) -> tuple[float, float]:
    a = (1 - level) / 2
    lo, hi = np.nanquantile(draws, [a, 1 - a])
    return float(lo), float(hi)


def holm(p) -> np.ndarray:
    """Holm-adjusted p-values, in the input order."""
    p = np.asarray(p, float)
    order = np.argsort(p)
    m = len(p)
    adj = np.maximum.accumulate((m - np.arange(m)) * p[order])
    out = np.empty(m)
    out[order] = np.minimum(adj, 1.0)
    return out

In [ ]:
%%writefile src/outcomes.py
"""Regular-season WR outcomes, per route and per player.

Separation at the throw depends heavily on things the receiver doesn't control:
man vs zone coverage (about 2.0 vs 3.4 yd on average), the route, and how long
the QB holds the ball. Separation over expected (`soe`) subtracts the average
separation of cohort WRs on the same route, against the same coverage, with a
similar time to throw. Screens are left out of separation outcomes: they are
designed to be open.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from src.routes import EXCLUDED_ROUTES

TTT_BINS = [0, 2.0, 2.5, 3.0, 3.5, np.inf]
TTT_LABELS = ["<2.0", "2.0-2.5", "2.5-3.0", "3.0-3.5", ">=3.5"]
MIN_CELL = 30  # routes; smaller cells fall back to a coarser cell


def separation_routes(routes: pd.DataFrame) -> pd.DataFrame:
    """Routes with a separation value and a throw time, with expected separation and `soe` added.

    Expected separation is the mean over (route, coverage, time-to-throw bin) when that cell has
    at least MIN_CELL routes, else over (route, coverage), else over route.
    """
    r = routes[routes.separation.notna() & routes.ttt.notna() & ~routes.route_ran.isin(EXCLUDED_ROUTES)].copy()
    r["ttt_bin"] = pd.cut(r.ttt, TTT_BINS, labels=TTT_LABELS, right=False).astype(str)
    expected = pd.Series(np.nan, index=r.index)
    for cell in (["route_ran", "coverage", "ttt_bin"], ["route_ran", "coverage"], ["route_ran"]):
        g = r.groupby(cell)["separation"]
        fill = expected.isna() & (g.transform("size") >= MIN_CELL)
        expected[fill] = g.transform("mean")[fill]
    r["expected_separation"] = expected.fillna(r.separation.mean())
    r["soe"] = r.separation - r.expected_separation
    return r


def explained_variance(sep_routes: pd.DataFrame) -> float:
    """Share of route-level separation variance explained by the expected-separation cells."""
    resid = sep_routes.separation - sep_routes.expected_separation
    return float(1 - resid.var() / sep_routes.separation.var())


def route_outcomes(routes: pd.DataFrame) -> pd.DataFrame:
    """Per-route outcome columns whose player mean is the player outcome.

    `target` (0/1) and `yards` average to target rate and yards per route run, and
    `epa_route` (EPA when targeted, else 0) to EPA per route run. `catch` is only set
    on targets and `yac_oe` only on catches, so their means are per target / per catch.
    """
    targeted = routes.target.fillna(False).astype(bool)
    caught = targeted & (routes.pass_result == "C")
    return routes.assign(
        target=targeted.astype(float),
        yards=routes.rec_yards.fillna(0),
        epa_route=routes.expected_points_added.where(targeted, 0.0),
        catch=caught.astype(float).where(targeted),
        yac_oe=(routes.yards_after_catch - routes.expected_yards_after_catch).where(caught),
    )


def player_outcomes(routes: pd.DataFrame, sep_routes: pd.DataFrame, prefix: str = "") -> pd.DataFrame:
    """One row per WR: usage, production and separation over the given routes."""
    r = route_outcomes(routes)
    g = r.groupby("nfl_id")
    targeted = r[r.target == 1]
    out = pd.DataFrame({
        "routes": g.size(),
        "games": g["game_id"].nunique(),
        "targets": g["target"].sum(),
        "rec_yards": g["rec_yards"].sum(min_count=1),
    })
    out["target_rate"] = out.targets / out.routes
    out["yprr"] = g["yards"].mean()
    out["epa_per_route"] = g["epa_route"].mean()
    out["epa_per_target"] = targeted.groupby("nfl_id")["expected_points_added"].mean()
    out["catch_rate"] = g["catch"].mean()
    out["yac_oe"] = g["yac_oe"].mean()
    s = sep_routes.groupby("nfl_id")
    out["sep_routes"] = s.size()
    out["separation"] = s["separation"].mean()
    out["soe"] = s["soe"].mean()
    out["man_share"] = s["coverage"].apply(lambda c: (c == "MAN_COVERAGE").mean())
    return out.add_prefix(prefix)


def career(con) -> pd.DataFrame:
    """Career usage and honors (player_career_successes), indexed by nfl_id."""
    return con.sql("SELECT * FROM player_career_successes").df().set_index("nfl_id")

In [ ]:
%%writefile src/routes.py
"""In-game WR routes: route windows, cuts with the Phase 1 detector, and player cut features.

The detector in `src.cuts` runs unchanged on game routes, so combine and game
cuts get identical processing. What changes is the scoring, because game breaks
vary more than drill breaks:

- A route window runs from the snap to 1 s after the throw. Cuts count if their
  apex is no later than 0.5 s after the throw: on timing routes the QB often
  releases just before the break. Cuts less than 1 yd past the receiver's snap
  position are release moves at the line, not route breaks, and are dropped.
- Cuts also need their whole 1.5 s entry window after the snap (apex >= 1.5 s).
  Earlier, entry speed and braking are measured partly on the receiver's stance
  or pre-snap motion, and the smoothing has no frames before the snap to work
  with: braking there tracks who goes in motion, not how a receiver cuts.
- Break side is relative to the field: `in` turns toward the middle of the field
  (a proxy for the ball, which isn't tracked), `out` toward the sideline. Combine
  route drills are run from a fixed side, so there turn direction already means
  the same thing.
- Slot = route_ran x break side. Within a slot, game breaks still differ in depth
  and angle (a 5-yd quick out vs a 12-yd deep out), which drives entry speed and
  speed retention. So each metric is regressed on depth and turn angle (with
  squares) within its slot and the standardized residual is the cut's score. In
  the combine each slot is one fixed drill break, so plain z-scores do the same job.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from src import cuts as C
from src.data import PARQUET

FIELD_CENTER_Y = 160 / 3 / 2   # yd; the field is 53.3 yd wide
TAIL = 1.0                     # s of tracking kept after the throw, so late breaks are fully measured
MAX_AFTER_THROW = 0.5          # s: latest apex that still counts as part of the route
MIN_DEPTH = 1.0                # yd past the snap position; shallower cuts are release moves
MIN_APEX_AFTER_SNAP = C.ENTRY_WINDOW * C.DT  # s: the entry window (1.5 s) must start after the snap
EXCLUDED_ROUTES = ("SCREEN",)  # designed separation, no route break
MIN_CUTS_PER_SLOT = 20
METRICS = ("entry_speed", "speed_retention", "peak_lat_accel", "peak_decel")
KEY = ["game_id", "play_id", "nfl_id"]


def _tracking_files() -> str:
    return ", ".join(f"'{PARQUET}/game_tracking_{y}.parquet'" for y in (2023, 2024, 2025))


def wr_routes(con) -> pd.DataFrame:
    """One row per WR route in a regular-season play (penalty-nullified plays dropped).

    Snap and throw times come from the receiver's own tracking event tags.
    `ttt` (time to throw, s) is null when the play has no pass_forward tag (sacks, scrambles).
    """
    return con.sql(f"""
        WITH ev AS (
            SELECT game_id, play_id, nfl_id,
                   min(time) FILTER (WHERE event = 'ball_snap') AS t_snap,
                   min(time) FILTER (WHERE event = 'pass_forward') AS t_pass
            FROM read_parquet([{_tracking_files()}])
            WHERE event IN ('ball_snap', 'pass_forward')
              AND nfl_id IN (SELECT nfl_id FROM players WHERE nfl_position = 'WR')
            GROUP BY ALL)
        SELECT pp.game_id, pp.play_id, pp.nfl_id, pp.season, pp.week, pp.route_ran,
               coalesce(pp.team_coverage_man_zone, 'UNKNOWN') AS coverage, pp.play_direction,
               coalesce(pp.in_motion_at_ball_snap, FALSE) AS in_motion,
               pp.pass_result, pp.target, pp.rec_yards, pp.yards_after_catch, pp.expected_yards_after_catch,
               pp.expected_points_added, pp.separation_at_pass_forward AS separation,
               ev.t_snap, ev.t_pass, (epoch_ms(ev.t_pass) - epoch_ms(ev.t_snap)) / 1000 AS ttt
        FROM player_play_reg pp
        JOIN players p USING (nfl_id)
        LEFT JOIN ev USING (game_id, play_id, nfl_id)
        WHERE p.nfl_position = 'WR' AND pp.route_ran IS NOT NULL AND pp.play_nullified_by_penalty = 'N'
        ORDER BY game_id, play_id, nfl_id
    """).df()


def route_frames(con, routes: pd.DataFrame) -> pd.DataFrame:
    """Tracking frames from the snap to TAIL seconds after the throw, for routes with both tags."""
    win = routes.loc[routes.t_snap.notna() & routes.t_pass.notna(), [*KEY, "t_snap", "t_pass"]]
    con.register("route_windows", win)
    try:
        return con.sql(f"""
            SELECT t.game_id, t.play_id, t.nfl_id, t.time, t.x, t.y, t.s
            FROM read_parquet([{_tracking_files()}]) t JOIN route_windows w USING (game_id, play_id, nfl_id)
            WHERE t.time BETWEEN w.t_snap AND w.t_pass + INTERVAL {int(TAIL * 1000)} MILLISECOND
            ORDER BY t.game_id, t.play_id, t.nfl_id, t.time
        """).df()
    finally:
        con.unregister("route_windows")


def break_side(turn_dir: str, vx_in: float, y_apex: float) -> str:
    """`in` if the turn rotates the incoming velocity toward the middle of the field, else `out`.

    Rotating velocity (vx, vy) counter-clockwise (a left turn) moves it toward the
    point (x, FIELD_CENTER_Y) when the cross product vx * (FIELD_CENTER_Y - y) is positive.
    """
    toward_middle_is_left = vx_in * (FIELD_CENTER_Y - y_apex) > 0
    return "in" if (turn_dir == "L") == toward_middle_is_left else "out"


def route_cuts(frames: pd.DataFrame, routes: pd.DataFrame) -> pd.DataFrame:
    """Every cut the Phase 1 detector finds in each route window, with route context.

    Adds `break_side`, `depth` (yd downfield from the snap position), `t_after_snap`,
    `t_after_throw` (s; negative = before the throw) and `kept` (counts as a route break).
    """
    info = {tuple(k): v for k, v in zip(routes[KEY].to_numpy(),
                                        routes[["route_ran", "play_direction", "ttt"]].itertuples(index=False))}
    f = frames.sort_values([*KEY, "time"])
    keys = f[KEY].to_numpy()
    starts = np.flatnonzero(np.r_[True, (keys[1:] != keys[:-1]).any(axis=1)])
    ends = np.r_[starts[1:], len(f)]
    xs, ys = f["x"].to_numpy(float), f["y"].to_numpy(float)
    out = []
    for s0, s1 in zip(starts, ends):
        if s1 - s0 <= 2 * C.HALF_WINDOW:
            continue
        key = tuple(keys[s0])
        x, y = xs[s0:s1], ys[s0:s1]
        k = C.kinematics(x, y)
        route, direction, ttt = info[key]
        downfield = 1.0 if direction == "right" else -1.0
        for i, c in enumerate(C.detect_cuts(k)):
            a = c["apex_frame"]
            b = max(a - C.HALF_WINDOW, 0)  # the incoming velocity the turn angle starts from
            out.append({
                **dict(zip(KEY, key)), "route_ran": route, "cut_idx": i, "n_frames": int(s1 - s0), **c,
                "break_side": break_side(c["turn_dir"], k["vx"][b], y[a]),
                "depth": downfield * (x[a] - x[0]),
                "t_after_snap": a * C.DT,
                "t_after_throw": a * C.DT - ttt,
            })
    cuts = pd.DataFrame(out)
    cuts["kept"] = (
        (cuts.t_after_throw <= MAX_AFTER_THROW + 1e-9)
        & (cuts.t_after_snap >= MIN_APEX_AFTER_SNAP - 1e-9)
        & (cuts.depth >= MIN_DEPTH)
        & ~cuts.route_ran.isin(EXCLUDED_ROUTES)
    )
    return cuts


def slot_z_adjusted(cuts: pd.DataFrame, metrics=METRICS) -> pd.DataFrame:
    """Add `slot` (route|side) and `<metric>_z`: within-slot standardized residual after depth and angle.

    Slots with fewer than MIN_CUTS_PER_SLOT cuts are dropped.
    """
    c = cuts.copy()
    c["slot"] = c["route_ran"] + "|" + c["break_side"]
    c = c[c.groupby("slot")["slot"].transform("size") >= MIN_CUTS_PER_SLOT].copy()
    for m in metrics:
        c[f"{m}_z"] = np.nan
    for _, g in c.groupby("slot"):
        X = np.column_stack([np.ones(len(g)), g.depth, g.depth ** 2, g.turn_deg, g.turn_deg ** 2])
        for m in metrics:
            yv = g[m].to_numpy(float)
            ok = np.isfinite(yv)
            beta, *_ = np.linalg.lstsq(X[ok], yv[ok], rcond=None)
            res = yv - X @ beta
            c.loc[g.index, f"{m}_z"] = res / np.nanstd(res[ok], ddof=1)
    return c


def main_breaks(zcuts: pd.DataFrame) -> pd.DataFrame:
    """Each route's main designed break: the first cut on the route's expected side (HITCH: first cut).

    Used as a sensitivity check; GO, FLAT, WHEEL and ANGLE routes have no single designed break and are left out.
    """
    expect = {"SLANT": "in", "IN": "in", "POST": "in", "CROSS": "in", "OUT": "out", "CORNER": "out"}
    z = zcuts.sort_values([*KEY, "apex_frame"])
    on_side = z.route_ran.map(expect) == z.break_side
    return pd.concat([z[on_side], z[z.route_ran == "HITCH"]]).groupby(KEY).head(1)


def player_features(zcuts: pd.DataFrame, prefix: str = "game", metrics=METRICS) -> pd.DataFrame:
    """One row per player: mean score per metric over all kept cuts, plus counts."""
    g = zcuts.groupby("nfl_id")
    feats = pd.DataFrame({
        f"{prefix}_n_cuts": g.size(),
        f"{prefix}_n_routes": zcuts.drop_duplicates(KEY).groupby("nfl_id").size(),
    })
    for m in metrics:
        feats[f"{prefix}_{m}"] = g[f"{m}_z"].mean()
    return feats

In [ ]:
%%writefile src/stats.py
"""Small statistics helpers shared across phases."""
from __future__ import annotations

import numpy as np
import pandas as pd
from scipy.stats import norm


def split_half(obs: pd.DataFrame, cols: list[str], unit: str = "game_id", n_iter: int = 200, seed: int = 0,
               min_units: int = 6, players=None) -> pd.DataFrame:
    """Spearman-Brown corrected split-half reliability of per-player means.

    `obs` has one row per observation (a cut, a route) with `nfl_id`, `unit` and `cols`.
    Each iteration splits every player's units (e.g. games) at random into two halves,
    averages each column per player within each half, and correlates the halves
    across players. Only `players` are used, if given; players with fewer than
    `min_units` units are left out.

    Restrict to players with enough volume: a few low-volume players have such
    noisy half-means that they drag the correlation down for everyone.
    """
    rng = np.random.default_rng(seed)
    d = obs if players is None else obs[obs.nfl_id.isin(players)]
    d = d[d.groupby("nfl_id")[unit].transform("nunique") >= min_units]
    units = d[["nfl_id", unit]].drop_duplicates()
    stats: dict[str, list[float]] = {c: [] for c in cols}
    for _ in range(n_iter):
        half = units.assign(h=rng.random(len(units)))
        half["h"] = half.groupby("nfl_id")["h"].rank(method="first") % 2
        dd = d.merge(half, on=["nfl_id", unit])
        a = dd[dd.h == 0].groupby("nfl_id")[cols].mean()
        b = dd[dd.h == 1].groupby("nfl_id")[cols].mean()
        for c in cols:
            r = a[c].corr(b[c])
            stats[c].append(2 * r / (1 + r) if r > -1 else np.nan)
    players = d["nfl_id"].nunique()
    return pd.DataFrame([{"feature": c, "reliability": np.nanmedian(v), "players": players} for c, v in stats.items()])


def detectable_r(n: int, alpha: float = 0.05, power: float = 0.8) -> float:
    """Smallest |Pearson r| a two-sided test detects with the given power at sample size n (Fisher z)."""
    return float(np.tanh((norm.ppf(1 - alpha / 2) + norm.ppf(power)) / np.sqrt(n - 3)))

In [ ]:
%%writefile notebooks/00_audit.py
# %% [markdown]
# # Phase 0 — data audit
#
# Loads every file, checks the data traps listed in PLAN.md §2, and writes
# counts per position / draft class / drill to `reports/00_audit.md`.
# Every `check(...)` is a hard assertion: if the data changes under us, this fails.
#
# Run: `python notebooks/00_audit.py` (opens as a notebook via jupytext percent format).

# %%
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))

import pandas as pd  # noqa: E402

from src.data import PARQUET, connect, to_parquet  # noqa: E402

to_parquet(overwrite=True)
con = connect()
report: list[str] = ["# Phase 0 — data audit", "", "Generated by `notebooks/00_audit.py`. Do not edit by hand.", ""]
checks: list[str] = []


def df(sql: str) -> pd.DataFrame:
    return con.sql(sql).df()


def one(sql: str):
    return con.sql(sql).fetchone()[0]


def check(name: str, ok: bool, detail: str = "") -> None:
    checks.append(f"- [{'x' if ok else ' '}] {name}" + (f" — {detail}" if detail else ""))
    assert ok, f"{name}: {detail}"


def md_table(d: pd.DataFrame) -> str:
    def fmt(v, col):
        if pd.isna(v):
            return ""
        if col == "season" or col.endswith("year"):
            return str(int(v)) if isinstance(v, (int, float)) else str(v)  # 2023, not 2,023
        if isinstance(v, float):
            return f"{v:,.0f}" if v == int(v) else f"{v:,.2f}"
        if isinstance(v, int):
            return f"{v:,}"
        return str(v)

    cols = [str(c) for c in d.columns]
    rows = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    rows += ["| " + " | ".join(fmt(v, c) for v, c in zip(r, cols)) + " |" for r in d.itertuples(index=False)]
    return "\n".join(rows)


def section(title: str, d: pd.DataFrame | None = None, note: str = "") -> None:
    report.extend([f"## {title}", ""])
    if note:
        report.extend([note, ""])
    if d is not None:
        report.extend([md_table(d), ""])
    print(f"\n== {title}\n{'' if d is None else d.to_string(index=False)}")


# %% [markdown]
# ## 1. Files load; keys are unique; every ID belongs to the 510-player cohort

# %%
counts = df(
    "SELECT * FROM ("
    + " UNION ALL ".join(
        f"SELECT '{p.stem}' AS file, count(*) AS rows FROM '{p}'" for p in sorted(PARQUET.glob("*.parquet"))
    )
    + ")"
)
section("Row counts", counts)

n_players = one("SELECT count(DISTINCT nfl_id) FROM players")
check("players.csv has 510 unique players", n_players == 510 == one("SELECT count(*) FROM players"))
for t in ["combine_results", "player_career_successes"]:
    check(f"{t}: one row per player, all in players.csv",
          one(f"SELECT count(DISTINCT nfl_id) FROM {t} WHERE nfl_id IN (SELECT nfl_id FROM players)")
          == one(f"SELECT count(*) FROM {t}") == 510)
check("combine_tracking: every player ID is in players.csv",
      one("""SELECT count(*) FROM combine_tracking t
             WHERE entity_type='PLAYER'
               AND NOT EXISTS (SELECT 1 FROM players p WHERE p.nfl_id = t.nfl_id)""") == 0)
check("game tracking: every nfl_id is a cohort rookie (no teammates/opponents)",
      one("""SELECT count(*) FROM (
                 SELECT DISTINCT nfl_id FROM game_tracking
             ) t WHERE NOT EXISTS (SELECT 1 FROM players p WHERE p.nfl_id = t.nfl_id)""") == 0)
check("player_play: every nfl_id is a cohort rookie",
      one("""SELECT count(*) FROM player_play pp
             WHERE NOT EXISTS (SELECT 1 FROM players p WHERE p.nfl_id = pp.nfl_id)""") == 0)
check("player_play: unique on (game_id, play_id, nfl_id)",
      one("SELECT count(*) FROM (SELECT 1 FROM player_play GROUP BY game_id, play_id, nfl_id HAVING count(*) > 1)") == 0)

# %% [markdown]
# ## 2. Cohort: position x draft class

# %%
section(
    "Players by draft class",
    df("""SELECT draft_year, count(*) AS players, count(draft_overall_pick) AS drafted,
                 count(*) - count(draft_overall_pick) AS udfa
          FROM players GROUP BY 1 ORDER BY 1"""),
    "UDFA = null `draft_overall_pick`.",
)
section(
    "Players by NFL position and draft class",
    df("""SELECT p.nfl_position, c.combine_position, count(*) AS total,
                 count(*) FILTER (WHERE p.draft_year = 2023) AS "2023",
                 count(*) FILTER (WHERE p.draft_year = 2024) AS "2024",
                 count(*) FILTER (WHERE p.draft_year = 2025) AS "2025"
          FROM players p JOIN combine_results c USING (nfl_id)
          GROUP BY ALL ORDER BY total DESC, 1"""),
    "No QBs, effectively no RBs, and no off-ball linebackers in the cohort.",
)

# %% [markdown]
# ## 3. Standard combine testing coverage

# %%
tests = ["ten_yd_split", "forty", "vertical", "broad_jump", "three_cone", "short_shuttle", "bench_reps",
         "ngs_athleticism_score"]
section(
    "Standard combine results: % of players with a value, by combine position",
    df("SELECT combine_position, count(*) AS n, "
       + ", ".join(f"round(100 * avg(({c} IS NOT NULL)::int)) AS {c}" for c in tests)
       + " FROM combine_results GROUP BY 1 ORDER BY n DESC"),
    "3-cone and shuttle are missing for most players, so a baseline needs imputation or must drop them.",
)

# %% [markdown]
# ## 4. Combine tracking: drills, reps, sampling

# %%
section(
    "Combine tracking by drill type",
    df("""SELECT drill_type, count(DISTINCT nfl_id) AS players, count(DISTINCT event_id) AS reps,
                 count(DISTINCT drill_name) AS drill_names, count(*) AS frames
          FROM combine_tracking WHERE entity_type = 'PLAYER' GROUP BY 1 ORDER BY 1"""),
    "One rep = one `event_id`. A few reps share `(nfl_id, drill_name, attempt)`, so key reps on `event_id`.",
)
section(
    "Players with tracking, by combine position x drill type",
    df("""PIVOT (SELECT c.combine_position, t.drill_type, t.nfl_id
                 FROM combine_tracking t JOIN combine_results c USING (nfl_id)
                 WHERE t.entity_type = 'PLAYER')
          ON drill_type USING count(DISTINCT nfl_id) GROUP BY combine_position ORDER BY combine_position"""),
)
section(
    "WR position drills (SKILL_DRILLS_WR)",
    df("""SELECT drill_name, count(DISTINCT nfl_id) AS players, count(DISTINCT event_id) AS reps,
                 round(count(*) / count(DISTINCT event_id), 1) AS avg_frames
          FROM combine_tracking WHERE drill_type = 'SKILL_DRILLS_WR' AND entity_type = 'PLAYER'
          GROUP BY 1 ORDER BY players DESC, drill_name"""),
)
check("combine_tracking: one player per event",
      one("SELECT max(n) FROM (SELECT count(DISTINCT nfl_id) n FROM combine_tracking WHERE entity_type='PLAYER' GROUP BY event_id)") == 1)
dt = df("""SELECT DISTINCT round(epoch_ms(time) - lag(epoch_ms(time)) OVER (PARTITION BY event_id, entity_type ORDER BY time)) AS dt
           FROM combine_tracking QUALIFY dt IS NOT NULL""")
check("combine_tracking: strictly 10 Hz, no gaps", dt["dt"].tolist() == [100.0], f"frame gaps (ms): {dt['dt'].tolist()}")

# %% [markdown]
# ## 5. Game tracking: season types and the REG-season filter

# %%
section(
    "games.csv",
    df("""SELECT season, season_type, count(*) AS games, min(week) AS wk_min, max(week) AS wk_max
          FROM games GROUP BY ALL ORDER BY 1, 2"""),
)
section(
    "Game tracking by season and season type (all files)",
    df("""SELECT season, season_type, count(DISTINCT game_id) AS games,
                 count(DISTINCT (game_id, play_id, nfl_id)) AS player_plays, count(*) AS frames,
                 round(100 * count(*) / sum(count(*)) OVER (PARTITION BY season), 1) AS pct_frames
          FROM game_tracking GROUP BY season, season_type ORDER BY 1, 2"""),
    "Preseason and playoff frames must be excluded; use the `game_tracking_reg` view.",
)
files = ", ".join(f"'{PARQUET}/game_tracking_{y}.parquet'" for y in (2023, 2024, 2025))
raw_frames = one(f"SELECT count(*) FROM read_parquet([{files}])")
check("game tracking: every game_id is in games.csv (no orphan frames)",
      raw_frames == one("SELECT count(*) FROM game_tracking"), f"{raw_frames:,} frames")
for y in (2023, 2024, 2025):
    check(f"game_tracking_{y}.csv holds season {y} only",
          one(f"SELECT count(*) FROM '{PARQUET}/game_tracking_{y}.parquet' t JOIN games g USING (game_id) WHERE g.season <> {y}") == 0)
check("game tracking: no duplicate frames",
      one(f"SELECT count(*) FROM (SELECT 1 FROM read_parquet([{files}]) GROUP BY game_id, play_id, nfl_id, time HAVING count(*) > 1)") == 0)
check("REG filter: game_tracking_reg has no PRE/POST frames",
      one("SELECT count(*) FROM game_tracking_reg g2 JOIN games USING (game_id) WHERE games.season_type <> 'REG'") == 0)
check("REG filter: August game IDs (e.g. 20240809xx) are all preseason",
      one("SELECT count(*) FROM games WHERE substr(game_id::varchar, 5, 2) = '08' AND season_type <> 'PRE'") == 0
      and one("SELECT count(*) FROM game_tracking_reg WHERE substr(game_id::varchar, 5, 2) = '08'") == 0)
check("REG filter: weeks 1-18 only",
      one("SELECT min(week) = 1 AND max(week) = 18 FROM game_tracking_reg"))

section(
    "player_play by season type",
    df("""SELECT g.season_type, count(*) AS rows FROM player_play JOIN games g USING (game_id)
          GROUP BY 1 ORDER BY 1"""),
    "player_play also includes PRE/POST; use the `player_play_reg` view.",
)
match = df("""WITH gp AS (SELECT DISTINCT game_id, play_id, nfl_id FROM game_tracking_reg)
              SELECT (SELECT count(*) FROM gp) AS tracked_player_plays,
                     (SELECT count(*) FROM gp JOIN player_play USING (game_id, play_id, nfl_id)) AS with_player_play_row,
                     (SELECT count(*) FROM player_play_reg) AS player_play_reg_rows""")
section("REG tracking <-> player_play overlap", match,
        "~1% of rows on either side have no partner; joins should be inner.")
check("REG tracking and player_play overlap >= 98%",
      match.with_player_play_row[0] / match.tracked_player_plays[0] >= 0.98
      and match.with_player_play_row[0] / match.player_play_reg_rows[0] >= 0.98)

# %% [markdown]
# ## 6. Exposure differs by draft class

# %%
section(
    "REG-season exposure by draft class x season",
    df("""SELECT p.draft_year, t.season, count(DISTINCT t.nfl_id) AS players,
                 count(DISTINCT (t.game_id, t.play_id, t.nfl_id)) AS player_plays
          FROM game_tracking_reg t JOIN players p USING (nfl_id) GROUP BY ALL ORDER BY 1, 2"""),
    "2023 class: three seasons; 2025 class: one. Normalize per snap/route and compare within class.",
)
section(
    "Players with any REG-season snap, by NFL position",
    df("""SELECT p.nfl_position, count(*) AS players, count(r.nfl_id) AS with_reg_snaps
          FROM players p LEFT JOIN (SELECT DISTINCT nfl_id FROM player_play_reg) r USING (nfl_id)
          GROUP BY 1 ORDER BY players DESC, nfl_position"""),
)

# %% [markdown]
# ## 7. WR focus: routes, targets, separation

# %%
section(
    "WRs: REG-season routes and separation coverage by draft class",
    df("""WITH r AS (SELECT nfl_id, count(route_ran) AS routes, sum(target::int) AS targets,
                            count(separation_at_pass_forward) AS sep_obs
                     FROM player_play_reg GROUP BY 1)
          SELECT p.draft_year, count(*) AS wrs, count(r.nfl_id) AS with_reg_snaps,
                 count(*) FILTER (WHERE routes >= 100) AS "100+ routes",
                 count(*) FILTER (WHERE routes >= 300) AS "300+ routes",
                 median(coalesce(routes, 0)) AS median_routes,
                 sum(routes) AS routes, sum(targets) AS targets, sum(sep_obs) AS separation_obs
          FROM players p LEFT JOIN r USING (nfl_id) WHERE p.nfl_position = 'WR'
          GROUP BY 1 ORDER BY 1"""),
    "`separation_at_pass_forward` is recorded on ~85% of routes (targeted or not), "
    "so it works as a per-route outcome, not just per target.",
)
section(
    "WR routes by type (REG)",
    df("""SELECT route_ran, count(*) AS routes, round(avg(separation_at_pass_forward), 2) AS mean_sep
          FROM player_play_reg pp JOIN players p USING (nfl_id)
          WHERE p.nfl_position = 'WR' AND route_ran IS NOT NULL GROUP BY 1 ORDER BY 2 DESC"""),
)

# %% [markdown]
# ## 8. Kinematics conventions (combine vs. game) — matters for Phase 1

# %%
conv = []
for name, src, part in [
    ("combine", "(SELECT * FROM combine_tracking WHERE entity_type = 'PLAYER')", "event_id"),
    ("game (2025 REG)", "(SELECT * FROM game_tracking_reg WHERE season = 2025)", "game_id, play_id, nfl_id"),
]:
    # a_tan: speed change (ds/dt); a_cen: turning (s * dθ/dt from dir). Both centered over ±1 frame.
    conv.append(df(f"""
        WITH t AS (SELECT *, lead(x) OVER w - x AS dx, lead(y) OVER w - y AS dy,
                          sqrt(power(x - lag(x) OVER w, 2) + power(y - lag(y) OVER w, 2)) AS step_back,
                          (lead(s) OVER w - lag(s) OVER w) / 0.2 AS a_tan,
                          s * radians(((lead(dir) OVER w - lag(dir) OVER w) + 540) % 360 - 180) / 0.2 AS a_cen
                   FROM {src} WINDOW w AS (PARTITION BY {part} ORDER BY time))
        SELECT '{name}' AS source,
               median(abs(((dir - degrees(atan2(dx, dy))) + 540) % 360 - 180)) AS dir_err_deg,
               median(abs(s - 10 * sqrt(dx * dx + dy * dy))) AS speed_vs_disp_err,
               median(dis / nullif(step_back, 0)) AS dis_vs_disp_ratio,
               count(*) FILTER (WHERE a < 0) AS negative_a_frames,
               corr(a, abs(a_tan)) AS a_corr_speed_change,
               corr(a, sqrt(a_tan * a_tan + a_cen * a_cen)) AS a_corr_total
        FROM t WHERE s > 2 AND dx IS NOT NULL"""))
conv = pd.concat(conv)
section(
    "Direction / speed / acceleration conventions",
    conv,
    "`dir` is degrees clockwise from +y in both sources, and `s` matches frame displacement. "
    "`a` is an unsigned magnitude in both (the few negatives in game data are glitches), and it is the "
    "*total* acceleration, speed change and turning combined: it tracks sqrt(a_tan² + a_cen²) "
    "(a_tan = ds/dt, a_cen = s·dθ/dt from `dir`) far better than |ds/dt| alone. So braking must be derived "
    "from smoothed speed and turn load from s·dθ/dt. The match is weaker in game data, so compute these the "
    "same way from x/y/s/dir in both sources rather than comparing the provided `a` across them. "
    "`dis` equals frame displacement in game tracking but not in combine tracking (next table). "
    "Combine tracking has no `o`.",
)
check("dir convention matches in combine and game tracking (median error < 5 deg)", (conv.dir_err_deg < 5).all())
by_source = conv.set_index("source")
check("game tracking: `dis` is the 0.1 s frame displacement (median ratio within 2% of 1)",
      abs(by_source.dis_vs_disp_ratio["game (2025 REG)"] - 1) < 0.02,
      f"{by_source.dis_vs_disp_ratio['game (2025 REG)']:.3f}")
check("combine tracking: `dis` is NOT the frame displacement (median ratio < 0.8); use `s` or x/y",
      by_source.dis_vs_disp_ratio["combine"] < 0.8, f"{by_source.dis_vs_disp_ratio['combine']:.2f}")
a_detail = ", ".join(f"{src}: r={tot:.2f} vs {spd:.2f}" for src, tot, spd in
                     conv[["source", "a_corr_total", "a_corr_speed_change"]].itertuples(index=False))
check("`a` is total acceleration: tracks sqrt(a_tan² + a_cen²) far better than |ds/dt| in both sources",
      (conv.a_corr_total > conv.a_corr_speed_change + 0.1).all(), a_detail)
section(
    "Combine `dis` vs frame displacement, by drill type",
    df("""WITH t AS (SELECT *, sqrt(power(x - lag(x) OVER w, 2) + power(y - lag(y) OVER w, 2)) AS step
                     FROM combine_tracking WHERE entity_type = 'PLAYER'
                     WINDOW w AS (PARTITION BY event_id ORDER BY time))
          SELECT drill_type, count(*) AS frames,
                 round(quantile_cont(dis / nullif(step, 0), 0.1), 2) AS p10,
                 round(median(dis / nullif(step, 0)), 2) AS median,
                 round(quantile_cont(dis / nullif(step, 0), 0.9), 2) AS p90
          FROM t WHERE s > 2 AND step IS NOT NULL GROUP BY 1 ORDER BY 1"""),
    "Ratio of combine `dis` to the actual 0.1 s displacement from x/y (frames with s > 2 yd/s). "
    "It is not a fixed scale factor: it varies by drill and is lowest in change-of-direction drills, "
    "so it cannot be rescaled. Use `s` or x/y for distance in combine data.",
)

# %%
report[4:4] = ["## Checks", "", *checks, ""]
out = ROOT / "reports" / "00_audit.md"
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text("\n".join(report), encoding="utf-8")
print(f"\n{len(checks)} checks passed; wrote {out.relative_to(ROOT)}")

In [ ]:
%%writefile notebooks/01_combine_features.py
# %% [markdown]
# # Phase 1 — combine cut features
#
# Detects cuts in every WR position-drill rep and every shuttle / 3-cone rep,
# validates the detector, measures how reliable each player-level feature is,
# and writes:
#
# - `data/features/combine_cuts.parquet`            one row per detected cut
# - `data/features/combine_player_features.parquet` one row per player
# - `reports/01_combine_features.md` + `reports/figures/01_*.png`
#
# Every `check(...)` is a hard assertion. Uses combine data only; no NFL outcomes.
#
# Run: `python notebooks/01_combine_features.py`

# %%
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))

import matplotlib  # noqa: E402

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
from matplotlib.collections import LineCollection  # noqa: E402
from matplotlib.colors import LinearSegmentedColormap  # noqa: E402

from src import cuts as C  # noqa: E402
from src import features as F  # noqa: E402
from src.data import connect  # noqa: E402

FEATURES = ROOT / "data" / "features"
FIGURES = ROOT / "reports" / "figures"
FEATURES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

con = connect()
report: list[str] = ["# Phase 1 — combine cut features", "",
                     "Generated by `notebooks/01_combine_features.py`. Do not edit by hand.", ""]
checks: list[str] = []


def check(name: str, ok: bool, detail: str = "") -> None:
    checks.append(f"- [{'x' if ok else ' '}] {name}" + (f" — {detail}" if detail else ""))
    assert ok, f"{name}: {detail}"


def md_table(d: pd.DataFrame, index: bool = False) -> str:
    if index:
        d = d.reset_index()

    def fmt(v):
        if isinstance(v, str):
            return v.replace("|", "\\|")  # slot names contain "|"
        if pd.isna(v):
            return ""
        if isinstance(v, (int, np.integer)) or (isinstance(v, float) and v == int(v) and abs(v) >= 10):
            return f"{int(v):,}"
        return f"{v:.2f}"

    rows = ["| " + " | ".join(map(str, d.columns)) + " |", "|" + "---|" * d.shape[1]]
    rows += ["| " + " | ".join(fmt(v) for v in r) + " |" for r in d.itertuples(index=False)]
    return "\n".join(rows)


def section(title: str, d: pd.DataFrame | None = None, note: str = "", index: bool = False) -> None:
    report.extend([f"## {title}", ""])
    if note:
        report.extend([note, ""])
    if d is not None:
        report.extend([md_table(d, index), ""])
    print(f"\n== {title}\n{'' if d is None else d.to_string()}")


# Chart styling: reference palette (dataviz skill), recessive axes.
SERIES = ["#2a78d6", "#eb6834", "#1baf7a"]
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
    "axes.edgecolor": GRID, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "axes.titlecolor": INK, "axes.titlesize": 10, "axes.labelsize": 9, "xtick.labelsize": 8, "ytick.labelsize": 8,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.6, "lines.linewidth": 2, "font.size": 9,
})

# %% [markdown]
# ## 1. Detect cuts

# %%
report.extend([
    "## Method", "",
    "- x/y are smoothed with a Savitzky–Golay filter (0.7 s, quadratic); velocity and acceleration come "
    "from its derivatives, split into along-track (braking / re-acceleration) and lateral (turning) parts. "
    "The same code will run on game routes in Phase 2.",
    "- A **cut** is a peak of at least 30° in the turn between the velocity 0.5 s before and 0.5 s after a frame. "
    "Its **apex** is the slowest frame near that peak. Cuts need ≥ 2 yd/s going in and ≥ 1.5 yd/s coming out.",
    "- Per cut: speed into the break (max over the prior 1.5 s), apex speed, speed retention (apex / entry), "
    "peak braking, distance used to slow down, peak lateral acceleration, speed regained 0.5 s after the apex.",
    "- Each metric is z-scored within its **slot** (the same break of the same drill across players, e.g. "
    "the curl's plant). A player's feature is their mean z, so players who ran different drills are comparable.",
    "",
])

frames = con.sql(
    """SELECT event_id, nfl_id, drill_type, drill_name, time, x, y, s FROM combine_tracking
       WHERE entity_type = 'PLAYER' AND drill_type IN ('SKILL_DRILLS_WR', 'SHORT_SHUTTLE', 'THREE_CONE_DRILL')
       ORDER BY event_id, time"""
).df()
err = np.concatenate([
    np.abs(C.kinematics(r.x.to_numpy(), r.y.to_numpy())["speed"] - r.s.to_numpy())
    for _, r in frames.groupby("event_id")
])
check("smoothed speed matches the provided `s`", np.median(err) < 0.1,
      f"median |diff| {np.median(err):.3f} yd/s, 95th pct {np.quantile(err, 0.95):.2f}")

pos_cuts = F.combine_cuts(con, F.POSITION_DRILLS)
agl_cuts = F.combine_cuts(con, F.AGILITY_DRILLS)
reps = frames.drop_duplicates("event_id").set_index("event_id")[["nfl_id", "drill_type", "drill_name"]]
all_cuts = pd.concat([pos_cuts, agl_cuts])
n_cuts = all_cuts.groupby("event_id").size().reindex(reps.index, fill_value=0)

per_rep = pd.crosstab(reps.drill_name, n_cuts.clip(upper=4).map(lambda k: f"{k}+" if k == 4 else str(k)))
per_rep.insert(0, "reps", per_rep.sum(axis=1))
section("Cuts detected per rep", per_rep, index=True,
        note="Straight-line drills (go route, over-the-shoulder, red-zone fade) should have none; "
             "the shuttle has two reversals; the 3-cone has two reversals plus three bends around cones.")

shuttle = reps.drill_name == "SHORT_SHUTTLE"
sharp = agl_cuts[agl_cuts.turn_deg >= 150].groupby("event_id").size().reindex(reps.index[shuttle], fill_value=0)
check("shuttle: most reps have exactly two ≥150° reversals", (sharp == 2).mean() >= 0.8,
      f"{(sharp == 2).mean():.0%} of {shuttle.sum()} reps")
go = reps.drill_name == "GO_ROUTE_RIGHT"
check("go route: no cuts on a straight sprint", (n_cuts[go] == 0).mean() >= 0.95,
      f"{(n_cuts[go] == 0).mean():.0%} of {go.sum()} reps")
check("3-cone: median of ≥ 4 cuts per rep",
      n_cuts[reps.drill_name == "THREE_CONE_DRILL"].median() >= 4)

# Turn direction must match route semantics (receiver's side of the formation).
# Reps with no detected cut count as misses, so missed detections can't hide.
first_dir = (pos_cuts.sort_values("apex_frame").groupby("event_id")["turn_dir"].first()
             .reindex(reps.index))
expect = {"CURL_ROUTE_RIGHT": "L", "COMEBACK_ROUTE_RIGHT": "R", "SLANT_ROUTE_LEFT": "R",
          "SPEED_OUT_ROUTE_LEFT": "L", "DAGGER_ROUTE_LEFT": "R", "SLOT_SAIL_ROUTE_RIGHT": "R"}
sem = pd.DataFrame([
    {"drill": d, "expected first break": f"{'left' if e == 'L' else 'right'}",
     "reps": int((reps.drill_name == d).sum()),
     "no cut detected": int(first_dir[reps.drill_name == d].isna().sum()),
     "share matching": (first_dir[reps.drill_name == d] == e).mean()}
    for d, e in expect.items()
])
section("First break direction vs route", sem,
        note="A right-side curl breaks back inside (left), a right-side comeback breaks to the sideline (right), "
             "a left-side slant breaks inside (right), and so on. Agreement confirms the left/right sign. "
             "Reps with no detected cut count as misses.")
check("first-break direction matches the route in ≥ 90% of reps, every route",
      (sem["share matching"] >= 0.9).all(), f"min {sem['share matching'].min():.0%}")

# %% [markdown]
# ## 2. Slot scoring and per-slot summaries

# %%
pos_z = F.slot_z(pos_cuts, by_direction=True)
agl_z = F.slot_z(agl_cuts, by_direction=False)
slot_tab = pos_z.groupby("slot").agg(
    cuts=("event_id", "size"), turn_deg=("turn_deg", "median"), entry_speed=("entry_speed", "median"),
    apex_speed=("apex_speed", "median"), retention=("speed_retention", "median"),
    peak_decel=("peak_decel", "median"), lat_accel=("peak_lat_accel", "median"),
    apex_time_s=("apex_frame", lambda s: s.median() * C.DT))
section("WR drill slots (medians)", slot_tab, index=True,
        note=f"{len(pos_z):,} of {len(pos_cuts):,} WR drill cuts fall in slots with ≥ {F.MIN_CUTS_PER_SLOT} cuts. "
             "Speeds in yd/s, accelerations in yd/s².")

# %% [markdown]
# ## 3. Does the detector measure agility? Shuttle / 3-cone features vs official times

# %%
cr = con.sql("""SELECT c.nfl_id, c.combine_position, c.forty, c.ten_yd_split, c.three_cone, c.short_shuttle,
                       c.combine_weight, c.combine_height, c.vertical, c.broad_jump
                FROM combine_results c""").df().set_index("nfl_id")
rows = []
for drill, official in [("SHORT_SHUTTLE", "short_shuttle"), ("THREE_CONE_DRILL", "three_cone")]:
    f = F.player_features(agl_z[agl_z.drill_name == drill], "f").join(cr)
    f = f[f[official].notna()]
    rows.append({"drill": drill, "players": len(f),
                 **{m: f[f"f_{m}"].corr(f[official]) for m in F.METRICS}})
validity = pd.DataFrame(rows)
section("Tracking features vs official stopwatch time (Pearson r)", validity,
        note="Negative = more of the feature goes with a faster official time. Players of all positions with both.")
for _, r in validity.iterrows():
    check(f"{r.drill}: harder braking goes with a faster official time", r.peak_decel <= -0.3, f"r = {r.peak_decel:.2f}")

# %% [markdown]
# ## 4. WR features: reliability

# %%
feats = F.player_features(pos_z, "pos").join(F.player_features(agl_z, "agl"), how="outer")
feats = cr[["combine_position"]].join(feats, how="inner")
rel = F.split_half_reliability(pos_z, n_iter=200)
kind = {"": "mean", "_lr": "left − right", "_sd": "consistency (SD)"}
rel["metric"] = rel.feature.str.replace(r"_(lr|sd)$", "", regex=True)
rel["kind"] = rel.feature.str.extract(r"(_lr|_sd)$")[0].fillna("").map(kind)
rel_tab = rel.pivot(index="metric", columns="kind", values="reliability").loc[list(F.METRICS),
                                                                              list(kind.values())]
section("Split-half reliability of WR position-drill features",
        rel_tab, index=True,
        note=f"{rel.players.iloc[0]} WRs with ≥ 4 drills; median over 200 random drill splits, Spearman–Brown "
             "corrected. Roughly: < 0.3 mostly noise, 0.5 usable with shrinkage, > 0.7 solid. "
             "Left − right is confounded with drill (each route breaks a fixed way) and consistency "
             "needs repeat reps the combine doesn't have; neither measures a stable trait here.")
PRIMARY = ["entry_speed", "speed_retention", "peak_lat_accel"]
check("pre-registered WR features have split-half reliability ≥ 0.45",
      (rel_tab.loc[PRIMARY, "mean"] >= 0.45).all(),
      ", ".join(f"{m} {rel_tab.loc[m, 'mean']:.2f}" for m in PRIMARY))

# %% [markdown]
# ## 5. WR features vs the stopwatch and each other

# %%
stop = ["forty", "ten_yd_split", "three_cone", "short_shuttle", "vertical", "broad_jump", "combine_weight",
        "combine_height"]
wr = feats[feats.combine_position == "WR"].join(cr[stop])
cols = [f"pos_{m}" for m in F.METRICS]
vs = pd.DataFrame({c: [wr[f].corr(wr[c]) for f in cols] for c in stop}, index=list(F.METRICS))
section("WR features vs standard combine results (Pearson r)", vs, index=True,
        note=f"{len(wr)} WRs; 3-cone and shuttle times exist for only ~1/3 of them. Small |r| against the "
             "40 means the cut features are not just straight-line speed again.")
section("WR feature intercorrelations", wr[cols].corr().set_axis(list(F.METRICS)).set_axis(list(F.METRICS), axis=1),
        index=True)

# %% [markdown]
# ## 6. Figures

# %%
# Figure 1: anatomy of one cut — a typical curl plant.
curl = pos_z[pos_z.slot == "CURL_ROUTE_RIGHT|L1"].copy()
curl["dist"] = curl[[f"{m}_z" for m in PRIMARY]].abs().sum(axis=1)
ex = curl.nsmallest(1, "dist").iloc[0]
r = frames[frames.event_id == ex.event_id]
k = C.kinematics(r.x.to_numpy(), r.y.to_numpy())
x, y = r.x.to_numpy() - r.x.iloc[0], r.y.to_numpy() - r.y.iloc[0]
# Rotate so the stem runs up the page.
ang = np.arctan2(x[15], y[15])
x, y = np.cos(ang) * x - np.sin(ang) * y, np.sin(ang) * x + np.cos(ang) * y
t = np.arange(len(r)) * C.DT
apex = int(ex.apex_frame)
entry = max(apex - C.ENTRY_WINDOW, 0) + int(np.argmax(k["speed"][max(apex - C.ENTRY_WINDOW, 0):apex + 1]))
after = apex + C.GAIN_FRAMES

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(10, 4.6), gridspec_kw={"width_ratios": [1, 1.6]})
pts = np.column_stack([x, y]).reshape(-1, 1, 2)
# Single-hue ramp, trimmed so the slowest frames stay visible on the light surface.
blues = LinearSegmentedColormap.from_list("blues", plt.get_cmap("Blues")(np.linspace(0.3, 1, 256)))
lc = LineCollection(np.concatenate([pts[:-1], pts[1:]], axis=1), cmap=blues, linewidth=3,
                    norm=plt.Normalize(0, k["speed"].max()))
lc.set_array(k["speed"][:-1])
ax0.add_collection(lc)
for i, label, off, ha in [(entry, "entry", (10, 0), "left"), (apex, "apex", (10, 4), "left"),
                          (after, "+0.5 s", (0, -16), "center")]:
    ax0.plot(x[i], y[i], "o", ms=8, mfc="#fcfcfb", mec=INK, mew=1.5)
    ax0.annotate(label, (x[i], y[i]), xytext=off, textcoords="offset points", color=INK, fontsize=9,
                 va="center", ha=ha)
ax0.set_aspect("equal")
ax0.autoscale()
ax0.margins(0.25)
ax0.set_title("Path (darker = faster)", loc="left")
ax0.set_xlabel("yards")
cb = fig.colorbar(lc, ax=ax0, shrink=0.7)
cb.set_label("speed (yd/s)", color=INK2)
cb.outline.set_visible(False)

ax1.plot(t, k["speed"], color=SERIES[0])
ax1.axvspan(t[entry], t[apex], color=GRID, alpha=0.6, lw=0)
for i, label in [(entry, f"entry {ex.entry_speed:.1f} yd/s"), (apex, f"apex {ex.apex_speed:.1f} yd/s"),
                 (after, f"+{ex.reaccel_gain:.1f} yd/s in 0.5 s")]:
    ax1.plot(t[i], k["speed"][i], "o", ms=8, mfc="#fcfcfb", mec=SERIES[0], mew=1.5)
    ax1.annotate(label, (t[i], k["speed"][i]), xytext=(8, 6), textcoords="offset points", color=INK, fontsize=9)
ax1.annotate(f"braking: peak {ex.peak_decel:.1f} yd/s²\nretention {ex.speed_retention:.0%}",
             ((t[entry] + t[apex]) / 2, 0.4), ha="center", color=INK2, fontsize=9)
ax1.set_xlabel("seconds into rep")
ax1.set_ylabel("speed (yd/s)")
ax1.set_ylim(bottom=0)
ax1.set_title("Speed through the break", loc="left")
fig.suptitle("Anatomy of a cut: a typical curl plant at the combine", x=0.01, ha="left", color=INK, fontsize=12)
fig.tight_layout()
fig.savefig(FIGURES / "01_cut_anatomy.png", dpi=110)
plt.close(fig)

# Figure 2: detected cuts on sample reps of every WR drill.
drills = sorted(pos_cuts.drill_name.unique())
rng = np.random.default_rng(7)
fig, axes = plt.subplots(3, 5, figsize=(15, 9.5))
for ax, d in zip(axes.ravel(), drills):
    evs = reps.index[reps.drill_name == d]
    for col, ev in zip(SERIES, rng.choice(evs, 3, replace=False)):
        rr = frames[frames.event_id == ev]
        xx, yy = rr.x.to_numpy() - rr.x.iloc[0], rr.y.to_numpy() - rr.y.iloc[0]
        a = np.arctan2(xx[15], yy[15])
        xx, yy = np.cos(a) * xx - np.sin(a) * yy, np.sin(a) * xx + np.cos(a) * yy
        ax.plot(xx, yy, color=col, lw=1.5)
        ap = pos_cuts.loc[pos_cuts.event_id == ev, "apex_frame"].astype(int)
        ax.plot(xx[ap], yy[ap], "o", ms=8, mfc="#fcfcfb", mec=col, mew=1.5)
    ax.set_aspect("equal", "datalim")
    ax.set_title(d.replace("_", " ").title(), fontsize=9, loc="left")
    ax.tick_params(labelsize=7)
for ax in axes.ravel()[len(drills):]:
    ax.remove()
fig.suptitle("Detected cut apexes (circles) on three random reps per WR drill; every path starts at (0, 0) "
             "heading up the page", x=0.01, ha="left", color=INK, fontsize=11)
fig.tight_layout()
fig.savefig(FIGURES / "01_cut_examples.png", dpi=90)
plt.close(fig)

report.extend([
    "## Figures", "",
    "![Anatomy of a cut](figures/01_cut_anatomy.png)", "",
    "![Detected cuts on sample reps](figures/01_cut_examples.png)", "",
])

# %%
pd.concat([pos_z, agl_z]).to_parquet(FEATURES / "combine_cuts.parquet", index=False)
feats.to_parquet(FEATURES / "combine_player_features.parquet")
report[4:4] = ["## Checks", "", *checks, ""]
out = ROOT / "reports" / "01_combine_features.md"
out.write_text("\n".join(report), encoding="utf-8")
print(f"\n{len(checks)} checks passed; {len(feats)} players; wrote {out.relative_to(ROOT)}")

In [ ]:
%%writefile notebooks/02_game_features.py
# %% [markdown]
# # Phase 2 — game cut features and outcomes
#
# Runs the Phase 1 cut detector, unchanged, on every regular-season WR route,
# validates it against route semantics, scores each break against breaks of the
# same route type, side, depth and angle, measures how stable each player-level
# game feature and outcome is, and writes:
#
# - `data/features/game_cuts.parquet`    one row per in-game route break, with scores
# - `data/features/game_routes.parquet`  one row per WR route, with expected separation
# - `data/features/wr_analysis.parquet`  one row per WR: draft, baseline tests, combine and game features, outcomes
#                                        (all seasons, and rookie season alone)
# - `reports/02_game_features.md` + `reports/figures/02_*.png`
#
# Every `check(...)` is a hard assertion. Phase 2 computes no correlation between combine
# features and game data, or between cut features and outcomes: those are Phase 3's
# pre-registered tests.
#
# Run: `python notebooks/02_game_features.py` (after `notebooks/01_combine_features.py`)

# %%
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))

import matplotlib  # noqa: E402

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
from matplotlib.lines import Line2D  # noqa: E402

from src import cuts as C  # noqa: E402
from src import features as F  # noqa: E402
from src import outcomes as O  # noqa: E402
from src import routes as R  # noqa: E402
from src.data import connect  # noqa: E402
from src.stats import detectable_r, split_half  # noqa: E402

FEATURES = ROOT / "data" / "features"
FIGURES = ROOT / "reports" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
if not (FEATURES / "combine_player_features.parquet").exists():
    raise SystemExit("Run notebooks/01_combine_features.py first: Phase 2 joins its combine features.")

con = connect()
report: list[str] = ["# Phase 2 — game cut features and outcomes", "",
                     "Generated by `notebooks/02_game_features.py`. Do not edit by hand.", ""]
checks: list[str] = []


def check(name: str, ok: bool, detail: str = "") -> None:
    checks.append(f"- [{'x' if ok else ' '}] {name}" + (f" — {detail}" if detail else ""))
    assert ok, f"{name}: {detail}"


def md_table(d: pd.DataFrame, index: bool = False) -> str:
    if index:
        d = d.reset_index()

    def fmt(v):
        if isinstance(v, str):
            return v.replace("|", "\\|")  # slot names contain "|"
        if pd.isna(v):
            return ""
        if isinstance(v, (int, np.integer)) or (isinstance(v, float) and v == int(v) and abs(v) >= 10):
            return f"{int(v):,}"
        return f"{round(float(v), 2) + 0.0:.2f}"  # + 0.0 turns -0.00 into 0.00

    rows = ["| " + " | ".join(map(str, d.columns)) + " |", "|" + "---|" * d.shape[1]]
    rows += ["| " + " | ".join(fmt(v) for v in r) + " |" for r in d.itertuples(index=False)]
    return "\n".join(rows)


def section(title: str, d: pd.DataFrame | None = None, note: str = "", index: bool = False) -> None:
    report.extend([f"## {title}", ""])
    if note:
        report.extend([note, ""])
    if d is not None:
        report.extend([md_table(d, index), ""])
    print(f"\n== {title}\n{'' if d is None else d.to_string()}")


def pct(v: float) -> str:
    return "" if pd.isna(v) else f"{v:.0%}"


# Chart styling: reference palette (dataviz skill), recessive axes. Same as Phase 1.
SERIES = ["#2a78d6", "#eb6834", "#1baf7a"]
SURFACE, INK, INK2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "axes.titlecolor": INK, "axes.titlesize": 10, "axes.labelsize": 9, "xtick.labelsize": 8, "ytick.labelsize": 8,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.6, "lines.linewidth": 2, "font.size": 9,
})

PRIMARY = ["entry_speed", "speed_retention", "peak_lat_accel"]  # pre-registered in Phase 1
NAMES = {"entry_speed": "speed into the break", "speed_retention": "speed retention (apex / entry)",
         "peak_lat_accel": "peak lateral (turning) acceleration", "peak_decel": "peak braking (secondary)"}
EXPECT = {"SLANT": "in", "IN": "in", "POST": "in", "CROSS": "in", "OUT": "out", "CORNER": "out"}
MIN_ROUTES = 100  # "qualified" WR: enough regular-season routes for stable player-level numbers

report.extend([
    "## Method", "",
    "- **Routes:** every WR route in a regular-season play (penalty-nullified plays dropped). The window runs "
    "from the snap to 1 s after the throw, both taken from the receiver's own tracking event tags.",
    "- **Cuts:** the Phase 1 detector runs unchanged, so game and combine breaks get identical smoothing, "
    "thresholds and metrics. A cut counts as a route break if its apex comes no later than 0.5 s after the throw "
    "(on timing routes the QB releases before the break), at least 1 yd past the receiver's snap position "
    "(shallower turns are release moves at the line), and at least 1.5 s after the snap, so the whole 1.5 s "
    "entry window is route running rather than the stance or pre-snap motion. Screens are left out: they have "
    "no route break.",
    "- **Break side:** `in` turns toward the middle of the field, `out` toward the sideline. The ball isn't tracked, "
    "so the middle of the field stands in for it.",
    "- **Scores:** slot = route × break side. Game breaks in a slot still differ in depth and angle (a 5-yd quick "
    "out vs a 12-yd deep out), which drives speed and retention. So within each slot every metric is regressed on "
    "depth and turn angle (with squares), and the standardized residual is the break's score. A player's game "
    "feature is the mean score over their breaks, the same construction as the combine features.",
    "- **Outcomes:** separation at the throw minus the cohort average for the same route, coverage (man / zone) "
    "and time to throw = separation over expected (SOE). Usage and production: target rate, yards per route run.",
    "- **Reliability:** split-half across games (each player's games split at random, 200 times, "
    f"Spearman–Brown corrected), for qualified WRs (≥ {MIN_ROUTES} regular-season routes). A few low-volume "
    "WRs have such noisy averages that including them drags every reliability down.",
    "- **Rookie season:** every game feature and outcome also exists for the rookie season alone, which every "
    "draft class has exactly once.",
    "",
])

# %% [markdown]
# ## 1. Route windows

# %%
routes = R.wr_routes(con)
has_window = routes.t_snap.notna() & routes.t_pass.notna()
qualified = routes.groupby("nfl_id").size().loc[lambda n: n >= MIN_ROUTES].index
frames = R.route_frames(con, routes)
win = pd.DataFrame({"routes": [
    len(routes), routes.t_snap.notna().sum(), routes.t_pass.notna().sum(), has_window.sum(),
    frames.groupby(R.KEY).ngroups,
]}, index=["WR routes in regular-season plays", "with a snap tag in the receiver's tracking",
           "with a throw (pass_forward) tag", "route windows (both tags)", "route windows with frames"]).rename_axis("step")
section("Route windows", win, index=True,
        note=f"Plays without a throw tag are mostly sacks and scrambles. Median time to throw "
             f"{routes.ttt.median():.1f} s; {len(frames):,} frames in the windows.")
check("≥ 99% of WR routes have a snap tag in the receiver's own tracking",
      routes.t_snap.notna().mean() >= 0.99, pct(routes.t_snap.notna().mean()))

sample = frames[R.KEY].drop_duplicates().sample(2000, random_state=0)
err = np.concatenate([
    np.abs(C.kinematics(g.x.to_numpy(), g.y.to_numpy())["speed"] - g.s.to_numpy())
    for _, g in frames.merge(sample, on=R.KEY).groupby(R.KEY) if len(g) > 2 * C.HALF_WINDOW
])
check("smoothed speed matches the provided `s` on game routes", np.median(err) < 0.1,
      f"median |diff| {np.median(err):.3f} yd/s over 2,000 random routes, 95th pct {np.quantile(err, 0.95):.2f}")

# %% [markdown]
# ## 2. Does the combine detector find real route breaks in games?

# %%
cuts = R.route_cuts(frames, routes)
kept = cuts[cuts.kept].copy()
n_win = routes[has_window].groupby("route_ran").size().drop(list(R.EXCLUDED_ROUTES))
first = kept.sort_values([*R.KEY, "apex_frame"]).groupby(R.KEY).first()
fg = first.groupby("route_ran")
side_ok = fg.apply(lambda d: (d.break_side == EXPECT[d.name]).mean() if d.name in EXPECT else np.nan,
                   include_groups=False)
per_route = pd.DataFrame({
    "routes": n_win,
    "with a break": (fg.size() / n_win).map(pct),
    "breaks per route": kept.groupby("route_ran").size() / n_win,
    "1st break: depth (yd)": fg.depth.median(),
    "1st break: turn (deg)": fg.turn_deg.median(),
    "1st break: s after snap": fg.t_after_snap.median(),
    "expected side": pd.Series(EXPECT),
    "1st break on expected side*": side_ok.map(pct),
}).sort_values("routes", ascending=False).fillna({"expected side": "—"}).rename_axis("route")
section("Breaks found per route type", per_route, index=True,
        note=f"{len(kept):,} of {len(cuts):,} detected cuts count as route breaks. Rounded breaks (many posts, "
             "corners and crossers) turn less than the detector's 30° in 0.5 s and aren't counted; hitches often "
             "end with the receiver settling rather than coming back. \\* Share of routes *with a break* whose "
             "first break goes the expected way; the share with a break is its own column, so misses stay visible. "
             "Routes with known geometry break the expected way, which validates the detector and the in / out labels.")
early = cuts.t_after_snap < R.MIN_APEX_AFTER_SNAP - 1e-9
motion = cuts[R.KEY].merge(routes[[*R.KEY, "in_motion"]], on=R.KEY, how="left").in_motion.astype(bool)
motion.index = cuts.index
why = pd.DataFrame({"cuts": [
    len(cuts), cuts.route_ran.isin(R.EXCLUDED_ROUTES).sum(), (cuts.t_after_throw > R.MAX_AFTER_THROW + 1e-9).sum(),
    (cuts.depth < R.MIN_DEPTH).sum(), early.sum(), len(kept),
]}, index=["detected in route windows", "on screens", "more than 0.5 s after the throw",
           "less than 1 yd past the snap position", "less than 1.5 s after the snap", "route breaks kept"])
section("Which detected cuts count as route breaks", why.rename_axis("step"), index=True,
        note="Reasons overlap, so the middle rows don't add up. Cuts in the first 1.5 s are mostly release moves "
             f"and receivers in motion turning upfield: {cuts[early].depth.lt(R.MIN_DEPTH).mean():.0%} are less "
             f"than 1 yd deep, and {motion[early].mean():.0%} come from a receiver in motion at the snap (vs "
             f"{motion[~early].mean():.0%} of later cuts). Their entry speed and braking are measured partly on the "
             "stance or the motion, so how often a WR has one reflects their role, not how they cut. The cost is "
             f"quick breaks: {early[cuts.route_ran == 'SLANT'].mean():.0%} of cuts detected on slants come this early.")
side_ok = side_ok.dropna()
check("among routes with a break, the first goes the route's way (in: slant, in, post, cross; out: out, corner) "
      "in ≥ 80%, every route", (side_ok >= 0.8).all(), ", ".join(f"{k} {v:.0%}" for k, v in side_ok.items()))

late = kept[(kept.t_after_throw > 0) & kept.route_ran.isin(list(EXPECT))]
late_first = late.sort_values([*R.KEY, "apex_frame"]).groupby(R.KEY).first()
late_ok = (late_first.break_side == late_first.route_ran.map(EXPECT)).mean()
check("breaks in the 0.5 s after the throw still go the route's way (≥ 85%): route breaks, not adjustments "
      "to the ball", late_ok >= 0.85, f"{late_ok:.0%} of {len(late_first):,} routes")

# %% [markdown]
# ## 3. Scoring breaks within slots

# %%
zc = R.slot_z_adjusted(kept)
zcols = [f"{m}_z" for m in R.METRICS]
slot_tab = zc.groupby("slot").agg(
    breaks=("slot", "size"), depth=("depth", "median"), turn_deg=("turn_deg", "median"),
    entry_speed=("entry_speed", "median"), retention=("speed_retention", "median"),
    lat_accel=("peak_lat_accel", "median"), braking=("peak_decel", "median"),
).sort_values("breaks", ascending=False)
section("Game slots (medians)", slot_tab, index=True,
        note=f"{len(zc):,} of {len(kept):,} route breaks fall in slots with ≥ {R.MIN_CUTS_PER_SLOT} breaks. "
             "Depth in yd downfield from the snap position; speeds in yd/s; accelerations in yd/s².")

plain = {m: (zc[m] - zc.groupby("slot")[m].transform("mean")) / zc.groupby("slot")[m].transform("std")
         for m in R.METRICS}
adj = pd.DataFrame({
    "depth: slot z": [plain[m].corr(zc.depth) for m in R.METRICS],
    "depth: adjusted": [zc[f"{m}_z"].corr(zc.depth) for m in R.METRICS],
    "turn: slot z": [plain[m].corr(zc.turn_deg) for m in R.METRICS],
    "turn: adjusted": [zc[f"{m}_z"].corr(zc.turn_deg) for m in R.METRICS],
}, index=[NAMES[m] for m in R.METRICS]).rename_axis("metric")
section("Correlation of break scores with depth and turn angle (Pearson r)", adj, index=True,
        note="Plain within-slot z-scores still carry route depth (deeper breaks are run faster) and angle "
             "(sharper breaks keep less speed). The adjusted scores don't.")
check("adjusted scores are uncorrelated with depth and turn angle (|r| < 0.05)",
      bool((adj[["depth: adjusted", "turn: adjusted"]].abs() < 0.05).all().all()))

# %% [markdown]
# ## 4. Are game cut features stable player traits?

# %%
gfeat = R.player_features(zc)
rel_all = split_half(zc, zcols, players=qualified).set_index("feature")
rel_main = split_half(R.main_breaks(zc), zcols, players=qualified).set_index("feature")
rel_game = pd.DataFrame({
    "all route breaks (primary)": rel_all.reliability.values,
    "main break only": rel_main.reliability.values,
}, index=[NAMES[m] for m in R.METRICS]).rename_axis("metric")
section("Split-half reliability of WR game cut features", rel_game, index=True,
        note=f"{rel_all.players.iloc[0]} qualified WRs (median "
             f"{zc[zc.nfl_id.isin(qualified)].groupby('nfl_id').size().median():.0f} breaks per WR). "
             "Main break only = each route's first break on its designed side (hitches: first "
             "break), a sensitivity check closest to a drill rep. Roughly: < 0.3 mostly noise, 0.5 usable with "
             "shrinkage, > 0.7 solid.")
check("game speed into the break is a stable player trait (split-half ≥ 0.6)",
      rel_all.reliability["entry_speed_z"] >= 0.6, f"{rel_all.reliability['entry_speed_z']:.2f}")

# %% [markdown]
# ## 5. Outcomes

# %%
sr = O.separation_routes(routes)
sep_tab = sr.pivot_table(index="route_ran", columns="coverage", values="separation", aggfunc="mean")
sep_tab = sep_tab[["MAN_COVERAGE", "ZONE_COVERAGE"]].rename(
    columns={"MAN_COVERAGE": "vs man", "ZONE_COVERAGE": "vs zone"}).rename_axis(None, axis=1)
sep_tab.insert(0, "routes", sr.groupby("route_ran").size())
sep_tab = sep_tab.rename_axis("route")
section("Mean separation at the throw (yd) by route and coverage",
        sep_tab.sort_values("routes", ascending=False), index=True,
        note=f"{len(sr):,} routes with separation and a throw time (screens left out). Expected-separation cells "
             f"(route × coverage × time-to-throw bin) explain {O.explained_variance(sr):.0%} of route-level "
             "variance; the rest is the receiver, the defender and the play.")

route_obs = O.route_outcomes(routes)
OUTCOME_OBS = {"separation": "separation", "soe": "soe", "target_rate": "target", "yprr": "yards",
               "epa_per_route": "epa_route", "catch_rate": "catch", "yac_oe": "yac_oe"}
rel_out = pd.concat([split_half(sr, ["separation", "soe"], players=qualified),
                     *[split_half(route_obs[route_obs[c].notna()], [c], players=qualified)
                       for c in list(OUTCOME_OBS.values())[2:]]]).set_index("feature")
q_out = O.player_outcomes(routes, sr).loc[lambda d: d.index.isin(qualified)]
out_tab = pd.DataFrame({
    "player value": ["mean separation at the throw (yd)", "separation over expected, SOE (yd)", "target rate",
                     "yards per route run", "EPA on targets, per route run", "catch rate (per target)",
                     "YAC over expected (yd per catch)"],
    "median": q_out[list(OUTCOME_OBS)].median().values,
    "split-half reliability": rel_out.reliability.loc[list(OUTCOME_OBS.values())].values,
}, index=list(OUTCOME_OBS)).rename_axis("outcome")
section("Outcomes and their reliability", out_tab, index=True,
        note=f"{len(q_out)} qualified WRs. SOE is the Phase 3 outcome; the rest are for reference. Catch rate and "
             "YAC over expected rest on targets and catches only, so they are the noisiest.")

# %% [markdown]
# ## 6. One table per WR for Phase 3

# %%
players = con.sql("""SELECT nfl_id, display_name, draft_year, draft_round, draft_overall_pick
                     FROM players WHERE nfl_position = 'WR'""").df().set_index("nfl_id")
baseline = con.sql("""SELECT nfl_id, combine_position, forty, ten_yd_split, vertical, broad_jump,
                             combine_height, combine_weight, ngs_athleticism_score
                      FROM combine_results""").df().set_index("nfl_id")
cfeat = pd.read_parquet(FEATURES / "combine_player_features.parquet")
cfeat = cfeat[["pos_n_cuts", "pos_n_drills", *[f"pos_{m}" for m in F.METRICS]]]
pout = O.player_outcomes(routes, sr)
rookie = routes.season == routes.nfl_id.map(players.draft_year)
rookie_keys = routes.loc[rookie, R.KEY]
pout_rookie = O.player_outcomes(routes[rookie], sr.merge(rookie_keys, on=R.KEY), prefix="rookie_")
gfeat_rookie = R.player_features(zc.merge(rookie_keys, on=R.KEY), prefix="game_rookie")
car = O.career(con)[["career_offensive_snaps", "career_games_active", "career_games_started"]]
games_with_breaks = zc.groupby("nfl_id").game_id.nunique().rename("game_n_games")
wr = (players.join(baseline).join(cfeat).join(gfeat).join(games_with_breaks).join(pout).join(car)
      .join(gfeat_rookie).join(pout_rookie))
wr.insert(wr.columns.get_loc("draft_overall_pick") + 1, "udfa", wr.draft_overall_pick.isna())

has_combine = wr.pos_entry_speed.notna()
enough_routes = wr.index.isin(qualified)
cov = pd.DataFrame({"WRs": [len(wr), has_combine.sum(), enough_routes.sum(), (has_combine & enough_routes).sum()]},
                   index=["WRs in the cohort", "with combine WR-drill features",
                          f"qualified (≥ {MIN_ROUTES} regular-season routes)",
                          "qualified, with combine features"]).rename_axis("group")
section("Who is in the Phase 3 table", cov, index=True)
check(f"every qualified WR (≥ {MIN_ROUTES} regular-season routes) has game cut features and SOE",
      bool(wr.loc[enough_routes, ["game_entry_speed", "soe"]].notna().all().all()))

by_class = wr.groupby("draft_year").agg(
    WRs=("display_name", "size"),
    qualified=("routes", lambda n: int((n >= MIN_ROUTES).sum())),
    with_rookie_routes=("rookie_routes", "count"),
    median_rookie_routes=("rookie_routes", "median"),
    median_rookie_breaks=("game_rookie_n_cuts", "median"),
)
by_class.index = by_class.index.astype(int).astype(str)  # a year, not a count: no thousands separator
section("Rookie-season coverage by draft class", by_class, index=True,
        note="The 2025 class has one season, so few of its WRs qualify on all-season volume. Rookie-season "
             "features and outcomes (`game_rookie_*`, `rookie_*` in `wr_analysis.parquet`) put every class on the "
             "same footing, at the cost of fewer breaks per WR.")

# %% [markdown]
# ## 7. What Phase 3 can detect (reliability only, no linkage)

# %%
ccuts = pd.read_parquet(FEATURES / "combine_cuts.parquet")
crel = F.split_half_reliability(ccuts[ccuts.drill_type == "SKILL_DRILLS_WR"], n_iter=200,
                                metrics=tuple(R.METRICS)).set_index("feature").reliability
n_link1 = n_soe = int((has_combine & enough_routes).sum())
rows = []
for m in R.METRICS:
    rows.append(("combine → game", NAMES[m], crel[m], rel_all.reliability[f"{m}_z"], n_link1))
for m in PRIMARY:
    rows.append(("combine → SOE", NAMES[m], crel[m], rel_out.reliability["soe"], n_soe))
det = pd.DataFrame(rows, columns=["link", "combine signal", "combine reliability", "game-side reliability", "WRs"])
det["ceiling on observed r"] = np.sqrt(det["combine reliability"].clip(lower=0) * det["game-side reliability"].clip(lower=0))
det["smallest detectable observed r"] = det.WRs.map(detectable_r)
true_r = det["smallest detectable observed r"] / det["ceiling on observed r"]
det["smallest detectable true r"] = true_r.map(lambda v: f"{v:.2f}" if v <= 1 else "> 1 (not detectable)")
section("Detectability of Phase 3's player-level tests", det,
        note="Measurement noise caps any observed correlation at √(reliability × reliability). With the WRs "
             "available, a two-sided test at α = 0.05 with 80% power needs the observed |r| in the 6th column, "
             "so the true correlation has to be at least the last column. This uses reliabilities only: no combine "
             "feature has been correlated with game data or outcomes. Route-level models of SOE (tens of "
             "thousands of routes, player random effects) are not limited the same way.")

# %% [markdown]
# ## 8. Figure

# %%
show = ["SLANT", "IN", "POST", "CROSS", "OUT", "CORNER", "HITCH", "GO"]
win_routes = routes[has_window]
fr = frames.set_index(R.KEY).sort_index()
kept_by = {k: g for k, g in kept.groupby(R.KEY)}
fig, axes = plt.subplots(2, 4, figsize=(14, 8.4))
for ax, rt in zip(axes.ravel(), show):
    for _, rr in win_routes[win_routes.route_ran == rt].sample(12, random_state=3).iterrows():
        key = (rr.game_id, rr.play_id, rr.nfl_id)
        g = fr.loc[key]
        t = (g.time - rr.t_snap).dt.total_seconds().to_numpy()
        n = int((t <= rr.ttt + R.MAX_AFTER_THROW + 1e-9).sum())
        x, y = g.x.to_numpy()[:n], g.y.to_numpy()[:n]
        d = 1.0 if rr.play_direction == "right" else -1.0
        up, right = d * (x - x[0]), -d * (y - y[0])
        if np.sign(R.FIELD_CENTER_Y - y[0]) != np.sign(-d):  # middle of the field on the receiver's left: mirror
            right = -right
        ax.plot(right, up, color=INK2, lw=1, alpha=0.55)
        if key in kept_by:
            for _, c in kept_by[key].iterrows():
                a = int(c.apex_frame)
                inside = c.break_side == "in"
                ax.plot(right[a], up[a], "o" if inside else "s", ms=7, mfc=SERIES[0] if inside else SERIES[1],
                        mec=SURFACE, mew=1.2, zorder=3)
    share = per_route.loc[rt, "with a break"]
    ax.set_title(f"{rt.title()} · break found on {share} of routes", loc="left")
    ax.set_aspect("equal", "datalim")
    ax.axvline(0, color=GRID, lw=1, zorder=0)
for ax in axes[:, 0]:
    ax.set_ylabel("downfield (yd)")
for ax in axes[1]:
    ax.set_xlabel("toward the middle of the field (yd)")
legend = [Line2D([], [], ls="", marker="o", ms=7, mfc=SERIES[0], mec=SURFACE, label="inside break"),
          Line2D([], [], ls="", marker="s", ms=7, mfc=SERIES[1], mec=SURFACE, label="outside break")]
fig.legend(handles=legend, loc="upper right", frameon=False, ncol=2, fontsize=9, labelcolor=INK)
fig.suptitle("Route breaks the combine detector finds in games: 12 random routes per route type, snap to 0.5 s "
             "after the throw.\nEvery path starts at (0, 0); downfield is up and the middle of the field is to the "
             "right.", x=0.01, ha="left", color=INK, fontsize=11)
fig.tight_layout(rect=(0, 0, 1, 0.95))
fig.savefig(FIGURES / "02_game_breaks.png", dpi=90)
plt.close(fig)
report.extend(["## Figure", "", "![Route breaks found in games](figures/02_game_breaks.png)", ""])

# %%
zc.to_parquet(FEATURES / "game_cuts.parquet", index=False)
routes.merge(sr[[*R.KEY, "ttt_bin", "expected_separation", "soe"]], on=R.KEY, how="left").to_parquet(
    FEATURES / "game_routes.parquet", index=False)
wr.to_parquet(FEATURES / "wr_analysis.parquet")
report[4:4] = ["## Checks", "", *checks, ""]
out = ROOT / "reports" / "02_game_features.md"
out.write_text("\n".join(report), encoding="utf-8")
print(f"\n{len(checks)} checks passed; {len(wr)} WRs; wrote {out.relative_to(ROOT)}")

In [ ]:
%%writefile notebooks/03_models.py
# %% [markdown]
# # Phase 3 — pre-registered models
#
# Runs exactly the analysis plan pre-registered in PLAN.md ("Phase 3 analysis plan", commit 6d770a1,
# committed before any combine–game or outcome link was computed):
#
# - Test 1, link (1): does each combine cut trait carry into the same trait in games, beyond the baseline?
# - Test 2, link (2): on routes where a WR cuts better than their own average, are they more open (SOE)?
# - Test 3, headline: does each combine cut trait predict SOE beyond the baseline? Plus a leave-one-draft-
#   class-out check of baseline vs baseline + the three signals.
#
# Writes `reports/03_models.md` + `reports/figures/03_*.png`. Every `check(...)` is a hard assertion.
#
# Run: `python notebooks/03_models.py` (after `notebooks/02_game_features.py`)

# %%
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))

import matplotlib  # noqa: E402

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
from matplotlib.lines import Line2D  # noqa: E402
from matplotlib.ticker import MaxNLocator  # noqa: E402

from src import features as F  # noqa: E402
from src import models as M  # noqa: E402
from src.stats import detectable_r, split_half  # noqa: E402

FEATURES = ROOT / "data" / "features"
FIGURES = ROOT / "reports" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
for f in ("wr_analysis.parquet", "game_cuts.parquet", "game_routes.parquet", "combine_cuts.parquet"):
    if not (FEATURES / f).exists():
        raise SystemExit(f"Missing {f}: run notebooks/01_combine_features.py and 02_game_features.py first.")

N_BOOT = 2000
MIN_ROUTES = 100
KEY = ["game_id", "play_id", "nfl_id"]
PRIMARY = ["entry_speed", "speed_retention", "peak_lat_accel"]
SECONDARY = ["peak_decel"]
SIGNALS = PRIMARY + SECONDARY
NAMES = {"entry_speed": "speed into the break", "speed_retention": "speed retention",
         "peak_lat_accel": "peak lateral acceleration", "peak_decel": "peak braking (secondary)"}

report: list[str] = ["# Phase 3 — pre-registered models", "",
                     "Generated by `notebooks/03_models.py`. Do not edit by hand.", ""]
checks: list[str] = []


def check(name: str, ok: bool, detail: str = "") -> None:
    checks.append(f"- [{'x' if ok else ' '}] {name}" + (f" — {detail}" if detail else ""))
    assert ok, f"{name}: {detail}"


def md_table(d: pd.DataFrame, index: bool = False) -> str:
    if index:
        d = d.reset_index()

    def fmt(v):
        if isinstance(v, str):
            return v.replace("|", "\\|")
        if pd.isna(v):
            return ""
        if isinstance(v, (int, np.integer)) or (isinstance(v, float) and v == int(v) and abs(v) >= 10):
            return f"{int(v):,}"
        return f"{round(float(v), 3) + 0.0:.3f}" if abs(v) < 0.1 else f"{round(float(v), 2) + 0.0:.2f}"

    rows = ["| " + " | ".join(map(str, d.columns)) + " |", "|" + "---|" * d.shape[1]]
    rows += ["| " + " | ".join(fmt(v) for v in r) + " |" for r in d.itertuples(index=False)]
    return "\n".join(rows)


def section(title: str, d: pd.DataFrame | None = None, note: str = "", index: bool = False) -> None:
    report.extend([f"## {title}", ""])
    if note:
        report.extend([note, ""])
    if d is not None:
        report.extend([md_table(d, index), ""])
    print(f"\n== {title}\n{'' if d is None else d.to_string()}")


def pfmt(p: float) -> str:
    return "< 0.001" if p < 0.001 else f"{p:.3f}"


def verdict(p_holm: float, p: float, yes: str) -> str:
    return yes if p_holm < 0.05 else ("suggestive" if p < 0.05 else "no evidence")


def ci_str(lo: float, hi: float, digits: int = 2) -> str:
    return f"[{lo:+.{digits}f}, {hi:+.{digits}f}]"


# Chart styling: same reference palette (dataviz skill) and recessive axes as Phases 1–2.
SERIES = ["#2a78d6", "#eb6834", "#1baf7a"]
SURFACE, INK, INK2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "axes.titlecolor": INK, "axes.titlesize": 12, "axes.labelsize": 11, "xtick.labelsize": 10, "ytick.labelsize": 11,
    "axes.spines.top": False, "axes.spines.right": False, "axes.spines.left": False, "axes.grid": True,
    "grid.color": GRID, "grid.linewidth": 0.6, "lines.linewidth": 2, "font.size": 11,
})

# %% [markdown]
# ## 0. Sample and baseline

# %%
wr = pd.read_parquet(FEATURES / "wr_analysis.parquet")
q = wr[wr.routes.fillna(0) >= MIN_ROUTES].copy()
check(f"62 qualified WRs (≥ {MIN_ROUTES} regular-season routes), as pre-registered", len(q) == 62, f"{len(q)}")
need = [f"pos_{m}" for m in SIGNALS] + [f"game_{m}" for m in SIGNALS] + ["soe"]
check("every qualified WR has all combine and game signals and SOE", bool(q[need].notna().all().all()))
B, B_medians = M.baseline_matrix(q)
miss = wr.loc[q.index, ["draft_overall_pick", "forty", "ten_yd_split", "vertical", "broad_jump"]].isna().sum()
report.extend([
    "## Method", "",
    "Exactly the plan pre-registered in PLAN.md (\"Phase 3 analysis plan\", commit `6d770a1`), which was committed "
    "before any combine–game or outcome link was computed. In short:",
    "",
    f"- **Sample:** the {len(q)} qualified WRs (≥ {MIN_ROUTES} regular-season routes); classes "
    + ", ".join(f"{int(k)}: {v}" for k, v in q.groupby("draft_year").size().items()) + ".",
    "- **Baseline (B):** log(draft pick) with UDFAs at pick 260 plus a UDFA flag, 40, 10-yd split, vertical, broad "
    f"jump, height, weight. Missing tests filled with the sample median ({miss['forty']} missing 40s, "
    f"{miss['vertical']} verticals, {miss['broad_jump']} broad jumps; {int(miss['draft_overall_pick'])} UDFAs).",
    "- **Test 1:** OLS of the game trait on B + the combine trait; HC3 p, player-bootstrap CI.",
    "- **Test 2:** route-level SOE on the route's mean break score split into within-player and player-mean parts; "
    "player-clustered p, player-cluster bootstrap CI.",
    "- **Test 3:** OLS of player SOE on B + the combine trait; HC3 p, bootstrap CI. Out of sample: ridge, leave one "
    "draft class out, B vs B + the three primary signals.",
    f"- Holm correction within each test's 3 primary signals; {N_BOOT:,} bootstrap resamples; braking and the "
    "other secondary analyses are not corrected.",
    "",
])

# %% [markdown]
# ## 1. Reliabilities (from Phases 1–2), for the disattenuated r and the power column

# %%
ccuts = pd.read_parquet(FEATURES / "combine_cuts.parquet")
crel = F.split_half_reliability(ccuts[ccuts.drill_type == "SKILL_DRILLS_WR"], n_iter=200,
                                metrics=tuple(SIGNALS)).set_index("feature").reliability
gcuts = pd.read_parquet(FEATURES / "game_cuts.parquet")
grel = split_half(gcuts, [f"{m}_z" for m in SIGNALS], players=q.index).set_index("feature").reliability
grel.index = grel.index.str.removesuffix("_z")
check("reliabilities match Phase 2 (game speed into the break 0.77)", abs(grel["entry_speed"] - 0.77) < 0.01,
      f"{grel['entry_speed']:.3f}")


def player_test(y: pd.Series, m: str, standardize_y: bool, sample: pd.DataFrame | None = None) -> dict:
    """OLS of y on the baseline + combine trait m (all standardized) over `sample` (default: the 62 qualified
    WRs); HC3 p and a player-bootstrap CI for the trait's coefficient."""
    sample = q if sample is None else sample
    Bs = B if sample is q else M.baseline_matrix(sample)[0]
    X = M.zscore(pd.concat([Bs, sample[f"pos_{m}"]], axis=1))
    yy = (y - y.mean()) / y.std(ddof=0) if standardize_y else y
    fit = M.ols(yy.to_numpy(), X.to_numpy())
    Xraw, yraw = pd.concat([Bs, sample[f"pos_{m}"]], axis=1).to_numpy(), y.to_numpy()

    def stat(idx):
        Xi = Xraw[idx]
        sd = Xi.std(axis=0)
        Xi = (Xi - Xi.mean(axis=0)) / np.where(sd > 0, sd, 1.0)
        yi = yraw[idx]
        yi = (yi - yi.mean()) / yi.std() if standardize_y else yi
        X1 = np.column_stack([np.ones(len(idx)), Xi])
        return np.linalg.lstsq(X1, yi, rcond=None)[0][-1]

    draws = M.bootstrap(stat, np.arange(len(sample)), N_BOOT, seed=0)
    lo, hi = M.percentile_ci(draws)
    return {"coef": fit["coef"][-1], "lo": lo, "hi": hi, "p": fit["p"][-1], "partial_r": fit["partial_r"][-1],
            "boot_fail": np.isnan(draws).mean()}


# %% [markdown]
# ## 2. Test 1 — link (1): combine trait → same trait in games

# %%
t1 = []
for m in SIGNALS:
    r = player_test(q[f"game_{m}"], m, standardize_y=True)
    raw = q[f"pos_{m}"].corr(q[f"game_{m}"])
    t1.append({"m": m, **r, "raw_r": raw, "disattenuated_r": raw / np.sqrt(crel[m] * grel[m])})
t1 = pd.DataFrame(t1).set_index("m")
t1["p_holm"] = np.nan
t1.loc[PRIMARY, "p_holm"] = M.holm(t1.loc[PRIMARY, "p"])
check("bootstrap fits succeed (≥ 99%)", (t1.boot_fail < 0.01).all())
det_r = detectable_r(len(q))
tab1 = pd.DataFrame({
    "combine reliability": crel[SIGNALS].values, "game reliability": grel[SIGNALS].values,
    "raw r": t1.raw_r.values, "disattenuated r": t1.disattenuated_r.values,
    "beyond baseline: β (SD per SD)": t1.coef.values,
    "95% CI": [ci_str(a, b) for a, b in zip(t1.lo, t1.hi)],
    "partial r": t1.partial_r.values,
    "p": [pfmt(v) for v in t1.p], "Holm p": ["" if pd.isna(v) else pfmt(v) for v in t1.p_holm],
    "verdict": [verdict(ph, p, "carries over") if m in PRIMARY else ("p < 0.05" if p < 0.05 else "no evidence")
                for m, ph, p in zip(t1.index, t1.p_holm, t1.p)],
}, index=[NAMES[m] for m in SIGNALS]).rename_axis("signal")
section("Test 1: which combine traits carry into games (translation table)", tab1, index=True,
        note=f"Game trait regressed on the baseline plus the combine trait, all standardized, {len(q)} WRs. β is the "
             "SD change in the game trait per SD of the combine trait with draft pick and athletic testing held "
             "fixed. Disattenuated r = raw r / √(combine reliability × game reliability): what the raw r would be "
             f"without measurement noise (descriptive, not tested). With {len(q)} WRs a test needs an observed |r| ≥ "
             f"{det_r:.2f} for 80% power.")

# %% [markdown]
# ## 3. Test 2 — link (2): break score → separation over expected, within player

# %%
routes = pd.read_parquet(FEATURES / "game_routes.parquet")
routes = routes[routes.nfl_id.isin(q.index) & routes.soe.notna()]
qcuts = gcuts[gcuts.nfl_id.isin(q.index)]
CLASS = q.draft_year


def route_scores(cuts: pd.DataFrame) -> pd.DataFrame:
    """One row per route with SOE and the mean break score per signal over the given breaks."""
    x = cuts.groupby(KEY)[[f"{m}_z" for m in SIGNALS]].mean().reset_index()
    return routes[[*KEY, "soe"]].merge(x, on=KEY)


def within_test(d: pd.DataFrame, cols: list[str], boot: bool = True) -> dict:
    """SOE on within-player and player-mean parts of route break scores (each in route-level SD units).

    Returns the within-player coefficient of the first column, with clustered p and a cluster-bootstrap CI.
    """
    d = d.copy()
    X = []
    for c in cols:
        x = d[c] / d[c].std()
        xbar = x.groupby(d.nfl_id).transform("mean")
        d[f"{c}_w"], d[f"{c}_b"] = x - xbar, xbar
        X += [f"{c}_w", f"{c}_b"]
    fit = M.ols(d.soe.to_numpy(), d[X].to_numpy(), cluster=d.nfl_id.to_numpy())
    out = {"b_w": fit["coef"][1], "p": fit["p"][1], "b_b": fit["coef"][2], "p_b": fit["p"][2],
           "routes": len(d), "WRs": d.nfl_id.nunique()}
    if boot:
        players = d.nfl_id.unique()
        rows = {p: np.flatnonzero(d.nfl_id.to_numpy() == p) for p in players}
        Xa = np.column_stack([np.ones(len(d)), d[X].to_numpy()])
        ya = d.soe.to_numpy()

        def stat(idx):
            r = np.concatenate([rows[players[i]] for i in idx])
            return np.linalg.lstsq(Xa[r], ya[r], rcond=None)[0][1]

        draws = M.bootstrap(stat, players, N_BOOT, seed=0)
        out["lo"], out["hi"] = M.percentile_ci(draws)
    return out


pre = route_scores(qcuts[qcuts.t_after_throw <= 1e-9])
check("route-level sample is the pre-registered ~8,400 routes", 8000 <= len(pre) <= 8800, f"{len(pre):,} routes")
t2 = pd.DataFrame({m: within_test(pre, [f"{m}_z"]) for m in SIGNALS}).T
t2["p_holm"] = np.nan
t2.loc[PRIMARY, "p_holm"] = M.holm(t2.loc[PRIMARY, "p"].astype(float))
tab2 = pd.DataFrame({
    "within-player effect (yd SOE per SD)": t2.b_w.astype(float).values,
    "95% CI": [ci_str(a, b, 3) for a, b in zip(t2.lo, t2.hi)],
    "p": [pfmt(v) for v in t2.p], "Holm p": ["" if pd.isna(v) else pfmt(v) for v in t2.p_holm],
    "verdict": [verdict(ph, p, "matters") if m in PRIMARY else ("p < 0.05" if p < 0.05 else "no evidence")
                for m, ph, p in zip(t2.index, t2.p_holm, t2.p)],
    "player-mean effect (yd per SD)": t2.b_b.astype(float).values,
    "player-mean p": [pfmt(v) for v in t2.p_b],
}, index=[NAMES[m] for m in SIGNALS]).rename_axis("signal")
section("Test 2: does the mechanic matter on the field?", tab2, index=True,
        note=f"{len(pre):,} routes from {pre.nfl_id.nunique()} WRs with a separation value and at least one break "
             "at or before the throw. Within-player effect: yd of separation over expected per 1 SD of the route's "
             "break score, comparing a WR's routes with each other, so the WR's role, QB and scheme cancel out. "
             "Player-mean effect: the same between WRs (confounded by those, reported only). "
             f"SOE has an SD of {pre.soe.std():.2f} yd per route.")

# %% [markdown]
# ## 4. Test 3 — headline: combine trait → SOE beyond the baseline

# %%
t3 = pd.DataFrame({m: player_test(q.soe, m, standardize_y=False) for m in SIGNALS}).T
t3["p_holm"] = np.nan
t3.loc[PRIMARY, "p_holm"] = M.holm(t3.loc[PRIMARY, "p"].astype(float))
tab3 = pd.DataFrame({
    "yd SOE per SD, beyond baseline": t3.coef.astype(float).values,
    "95% CI": [ci_str(a, b, 3) for a, b in zip(t3.lo, t3.hi)],
    "partial r": t3.partial_r.astype(float).values,
    "p": [pfmt(v) for v in t3.p], "Holm p": ["" if pd.isna(v) else pfmt(v) for v in t3.p_holm],
    "verdict": [verdict(ph, p, "predicts") if m in PRIMARY else ("p < 0.05" if p < 0.05 else "no evidence")
                for m, ph, p in zip(t3.index, t3.p_holm, t3.p)],
}, index=[NAMES[m] for m in SIGNALS]).rename_axis("signal")
section("Test 3: does the combine trait predict separation beyond the baseline?", tab3, index=True,
        note=f"Player SOE (yd, mean over each WR's routes; SD across WRs {q.soe.std():.2f} yd) regressed on the "
             "baseline plus the combine trait (standardized).")


def build(cols):
    def _build(train, test):
        Btr, med = M.baseline_matrix(train)
        Bte, _ = M.baseline_matrix(test, med)
        Xtr = pd.concat([Btr, train[cols]], axis=1)
        Xte = pd.concat([Bte, test[cols]], axis=1)
        return M.zscore(Xtr), M.zscore(Xte, ref=Xtr)
    return _build


lo_rows = []
for label, cols in [("baseline", []), ("baseline + 3 combine cut signals", [f"pos_{m}" for m in PRIMARY])]:
    pred = M.leave_one_group_out(q, "soe", build(cols), "draft_year")
    row = {"model": label, "pooled out-of-sample R²": M.oos_r2(q.soe, pred)}
    for c in sorted(q.draft_year.unique()):
        k = q.draft_year == c
        row[f"MSE, {int(c)} held out"] = float(np.mean((q.soe[k] - pred[k]) ** 2))
    lo_rows.append(row)
lodo = pd.DataFrame(lo_rows).set_index("model")
check("each draft class is held out once", lodo.filter(like="held out").shape[1] == 3)
gain = lodo.iloc[1, 0] - lodo.iloc[0, 0]
section("Test 3, out of sample: leave one draft class out", lodo, index=True,
        note="Ridge regression (penalty chosen by generalized cross-validation inside the two training classes), "
             f"predicting the held-out class's player SOE. Gain from adding the three signals: {gain:+.3f} R². "
             "Negative R² means the model predicts held-out WRs worse than their class-pooled mean would.")

base_fit = M.ols(q.soe.to_numpy(), M.zscore(B).to_numpy())
base_tab = pd.DataFrame({"yd SOE per SD": base_fit["coef"][1:], "p": [pfmt(v) for v in base_fit["p"][1:]]},
                        index=B.columns).rename_axis("baseline term")
section("Exploratory (not pre-registered): what the baseline picks up in SOE", base_tab, index=True,
        note="Added after the pre-registered tests ran, to explain the baseline's out-of-sample R². Weight carries "
             "it: heavier WRs are less open. SOE adjusts for route, coverage and time to throw but not alignment, so "
             "this is probably role (bigger outside receivers vs lighter slot receivers) rather than a cutting effect; "
             "that explanation is not tested here.")

# %% [markdown]
# ## 5. Secondary analyses (pre-registered, not corrected)

# %%
sec = []
for outcome, label in [("target_rate", "target rate"), ("yprr", "yards per route run")]:
    for m in PRIMARY:
        r = player_test(q[outcome], m, standardize_y=True)
        sec.append({"analysis": f"Test 3 on {label} (SD per SD)", "signal": NAMES[m], "estimate": r["coef"],
                    "95% CI": ci_str(r["lo"], r["hi"]), "p": pfmt(r["p"])})
qr_mask = wr.rookie_routes.fillna(0) >= MIN_ROUTES
qr = wr[qr_mask & wr.rookie_soe.notna()]
for m in PRIMARY:
    r = player_test(qr.rookie_soe, m, standardize_y=False, sample=qr)
    sec.append({"analysis": f"Test 3 on rookie-season SOE (yd per SD; {len(qr)} WRs)", "signal": NAMES[m],
                "estimate": r["coef"], "95% CI": ci_str(r["lo"], r["hi"], 3), "p": pfmt(r["p"])})
allk = route_scores(qcuts)
for m in PRIMARY:
    r = within_test(allk, [f"{m}_z"])
    sec.append({"analysis": f"Test 2 with all kept breaks ({len(allk):,} routes)", "signal": NAMES[m],
                "estimate": r["b_w"], "95% CI": ci_str(r["lo"], r["hi"], 3), "p": pfmt(r["p"])})
for c in sorted(q.draft_year.unique()):
    d = pre[pre.nfl_id.map(CLASS) == c]
    for m in PRIMARY:
        r = within_test(d, [f"{m}_z"], boot=False)
        sec.append({"analysis": f"Test 2, {int(c)} class only ({r['WRs']} WRs)", "signal": NAMES[m],
                    "estimate": r["b_w"], "95% CI": "", "p": pfmt(r["p"])})
sec = pd.DataFrame(sec)
section("Secondary analyses", sec,
        note="Pre-registered as secondary: reported for context, no multiple-testing correction, no verdicts.")

# %% [markdown]
# ## 6. Figure: the three tests side by side

# %%
fig, axes = plt.subplots(1, 3, figsize=(13, 4.6), sharey=True)
panels = [
    (t1, "coef", "Test 1 · combine → game trait\nβ beyond baseline (SD per SD)", "p_holm"),
    (t2, "b_w", "Test 2 · break score → separation\nwithin-player effect (yd SOE per SD)", "p_holm"),
    (t3, "coef", "Test 3 · combine trait → separation\nbeyond baseline (yd SOE per SD)", "p_holm"),
]
ypos = np.arange(len(SIGNALS))[::-1]
for ax, (t, col, title, ph) in zip(axes, panels):
    for y, m in zip(ypos, SIGNALS):
        est, lo, hi = float(t.loc[m, col]), float(t.loc[m, "lo"]), float(t.loc[m, "hi"])
        color = SERIES[0] if m in PRIMARY else INK2
        sig = m in PRIMARY and float(t.loc[m, ph]) < 0.05
        ax.plot([lo, hi], [y, y], color=color, lw=2, solid_capstyle="round")
        ax.plot(est, y, "o", ms=8, mfc=color if sig else SURFACE, mec=color, mew=2, zorder=3)
    ax.axvline(0, color=INK2, lw=1, zorder=0)
    ax.xaxis.set_major_locator(MaxNLocator(5))
    ax.set_title(title, loc="left")
    ax.grid(axis="y", visible=False)
    ax.tick_params(axis="y", length=0)
axes[0].set_yticks(ypos, [NAMES[m] for m in SIGNALS])
legend = [Line2D([], [], ls="", marker="o", ms=8, mfc=SERIES[0], mec=SERIES[0], mew=2,
                 label="primary signal, Holm p < 0.05"),
          Line2D([], [], ls="", marker="o", ms=8, mfc=SURFACE, mec=SERIES[0], mew=2,
                 label="primary signal, not significant after Holm"),
          Line2D([], [], ls="", marker="o", ms=8, mfc=SURFACE, mec=INK2, mew=2, label="secondary (not corrected)")]
fig.legend(handles=legend, loc="lower center", ncol=3, frameon=False, fontsize=10, labelcolor=INK)
fig.suptitle("Only the in-game break moves separation; the combine traits predict neither the game trait nor "
             "separation", x=0.01, ha="left", color=INK, fontsize=13)
fig.text(0.01, 0.875, "Estimates with 95% bootstrap intervals, 62 qualified WRs", ha="left", color=INK2, fontsize=10)
fig.tight_layout(rect=(0, 0.08, 1, 0.87))
fig.savefig(FIGURES / "03_tests.png", dpi=150)
plt.close(fig)
report.extend(["## Figure", "", "![The three pre-registered tests](figures/03_tests.png)", ""])

# %%
report[4:4] = ["## Checks", "", *checks, ""]
out = ROOT / "reports" / "03_models.md"
out.write_text("\n".join(report), encoding="utf-8")
print(f"\n{len(checks)} checks passed; wrote {out.relative_to(ROOT)}")

In [ ]:
%%writefile notebooks/03b_sensitivity.py
# %% [markdown]
# # Sensitivity checks and a positive control (after Phase 3, not pre-registered)
#
# Added after the pre-registered Phase 3 tests ran. Nothing here replaces them; every result is reported
# whichever way it comes out, and each rule was fixed before the combine–game link it feeds was computed.
#
# 1. **Matched combine cuts.** The Phase 1 combine features average every WR drill cut, including turns the
#    receiver makes after the catch and breaks less than 1.5 s into the rep. The game features count neither
#    (breaks must come ≤ 0.5 s after the throw and ≥ 1.5 s after the snap). Here the combine features are rebuilt
#    with the game's rules, timing each drill break against the throw from the combine ball tracking, and
#    Tests 1 and 3 are rerun on them.
# 2. **Positive control.** Does the pipeline find transfer where it should exist? Top speed in the tracked
#    40-yard dash vs peak speed on regular-season go routes.
# 3. **Play context.** Test 2's within-receiver effect removes the receiver's role, QB and scheme, but not
#    play-level context. Press coverage (cushion), alignment, field position, down and distance, game state,
#    the coverage shell and the route's design could each move both break speed and separation. Test 2 is
#    rerun with each group added, all together, and all together within the same game.
# 4. **Man vs zone.** Receivers are coached to win with speed through the break against man and to settle into
#    open grass against zone. Prediction, stated before computing: speed into the break matters more against man.
# 5. **What a yard is worth.** Separation in football units: completion %, EPA per target and the chance of being
#    targeted per yard of separation, pooled and by coverage, then applied to the effects above.
#
# Writes `reports/03b_sensitivity.md`, `data/features/sensitivity.json` with the positive control for the
# writeup's Figure 4, and `data/features/combine_matched_features.parquet` for the robustness note on Figure 5.
# Every `check(...)` is a hard assertion.
#
# Run: `python notebooks/03b_sensitivity.py` (after `notebooks/03_models.py`)

# %%
import json
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))

import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402

from src import cuts as C  # noqa: E402
from src import data  # noqa: E402
from src import features as F  # noqa: E402
from src import models as M  # noqa: E402
from src import routes as R  # noqa: E402
from src.stats import split_half  # noqa: E402

FEATURES = ROOT / "data" / "features"
for f in ("wr_analysis.parquet", "game_cuts.parquet", "game_routes.parquet", "combine_cuts.parquet"):
    if not (FEATURES / f).exists():
        raise SystemExit(f"Missing {f}: run notebooks/01_combine_features.py and 02_game_features.py first.")

N_BOOT = 2000
MIN_ROUTES = 100
PRIMARY = ["entry_speed", "speed_retention", "peak_lat_accel"]
NAMES = {"entry_speed": "speed into the break", "speed_retention": "speed retention",
         "peak_lat_accel": "peak lateral acceleration"}
BALL_THROWN = 8.0          # yd/s: the ball is in flight once it moves faster than this
MIN_BALL_REPS = 10         # reps with ball tracking needed to time a slot against the throw
TWIN_SLOTS = {"SPEED_OUT_LEFT|L1": "SPEED_OUT_ROUTE_LEFT|L1"}  # same break, drill logged under two names
MIN_GO_ROUTES = 10

report: list[str] = ["# Sensitivity checks and a positive control", "",
                     "Generated by `notebooks/03b_sensitivity.py`. Do not edit by hand.", "",
                     "**Not pre-registered.** Added after the Phase 3 tests ran; the pre-registered results in "
                     "`reports/03_models.md` stand as they are. Each rule below was fixed before the combine–game "
                     "link it feeds was computed, and every result is reported.", ""]
checks: list[str] = []


def check(name: str, ok: bool, detail: str = "") -> None:
    checks.append(f"- [{'x' if ok else ' '}] {name}" + (f" — {detail}" if detail else ""))
    assert ok, f"{name}: {detail}"


def md_table(d: pd.DataFrame, index: bool = False) -> str:
    if index:
        d = d.reset_index()

    def fmt(v):
        if isinstance(v, str):
            return v.replace("|", "\\|")
        if pd.isna(v):
            return ""
        if isinstance(v, (bool, np.bool_)):
            return "yes" if v else "no"
        if isinstance(v, (int, np.integer)) or (isinstance(v, float) and v == int(v) and abs(v) >= 10):
            return f"{int(v):,}"
        return f"{round(float(v), 3) + 0.0:.3f}" if abs(v) < 0.1 else f"{round(float(v), 2) + 0.0:.2f}"

    rows = ["| " + " | ".join(map(str, d.columns)) + " |", "|" + "---|" * d.shape[1]]
    rows += ["| " + " | ".join(fmt(v) for v in r) + " |" for r in d.itertuples(index=False)]
    return "\n".join(rows)


def section(title: str, d: pd.DataFrame | None = None, note: str = "", index: bool = False) -> None:
    report.extend([f"## {title}", ""])
    if note:
        report.extend([note, ""])
    if d is not None:
        report.extend([md_table(d, index), ""])
    print(f"\n== {title}\n{'' if d is None else d.to_string()}")


def pfmt(p: float) -> str:
    return "< 0.001" if p < 0.001 else f"{p:.3f}"


def ci_str(lo: float, hi: float, digits: int = 2) -> str:
    return f"[{lo:+.{digits}f}, {hi:+.{digits}f}]"


def player_test(sample: pd.DataFrame, y: str, x: str, standardize_y: bool) -> dict:
    """Phase 3's player-level test: OLS of y on the baseline + x (all standardized); HC3 p, bootstrap CI."""
    B, _ = M.baseline_matrix(sample)
    Xraw = pd.concat([B, sample[x]], axis=1)
    yv = sample[y]
    yy = (yv - yv.mean()) / yv.std(ddof=0) if standardize_y else yv
    fit = M.ols(yy.to_numpy(), M.zscore(Xraw).to_numpy())
    Xa, ya = Xraw.to_numpy(), yv.to_numpy()

    def stat(idx):
        Xi = Xa[idx]
        sd = Xi.std(axis=0)
        Xi = (Xi - Xi.mean(axis=0)) / np.where(sd > 0, sd, 1.0)
        yi = ya[idx]
        yi = (yi - yi.mean()) / yi.std() if standardize_y else yi
        return np.linalg.lstsq(np.column_stack([np.ones(len(idx)), Xi]), yi, rcond=None)[0][-1]

    lo, hi = M.percentile_ci(M.bootstrap(stat, np.arange(len(sample)), N_BOOT, seed=0))
    return {"coef": fit["coef"][-1], "lo": lo, "hi": hi, "p": fit["p"][-1]}


def r_ci(a: pd.Series, b: pd.Series) -> tuple[float, float, float]:
    """Pearson r with a player-bootstrap 95% CI."""
    av, bv = a.to_numpy(), b.to_numpy()
    draws = M.bootstrap(lambda i: np.corrcoef(av[i], bv[i])[0, 1], np.arange(len(av)), N_BOOT, seed=0)
    return float(np.corrcoef(av, bv)[0, 1]), *M.percentile_ci(draws)


con = data.connect()
wr = pd.read_parquet(FEATURES / "wr_analysis.parquet")
q = wr[wr.routes.fillna(0) >= MIN_ROUTES].copy()
check("62 qualified WRs, as in Phase 3", len(q) == 62, f"{len(q)}")

# %% [markdown]
# ## 1. Matched combine cuts: the game's inclusion rules applied to the drills

# %%
# Throw time in s from the player's first frame, the same clock as the cut apex.
throws = con.sql(f"""
    WITH p AS (SELECT event_id, min(time) AS t0 FROM combine_tracking
               WHERE drill_type = 'SKILL_DRILLS_WR' AND entity_type = 'PLAYER' GROUP BY 1)
    SELECT b.event_id, (epoch_ms(min(b.time)) - epoch_ms(any_value(p.t0))) / 1000.0 AS t_throw
    FROM combine_tracking b JOIN p USING (event_id)
    WHERE b.entity_type = 'BALL' AND b.s > {BALL_THROWN} GROUP BY 1""").df()
ball_reps = con.sql("""SELECT count(DISTINCT event_id) FROM combine_tracking
                       WHERE drill_type = 'SKILL_DRILLS_WR' AND entity_type = 'BALL'""").fetchone()[0]
check("a throw is found in ≥ 95% of WR drill reps with ball tracking", len(throws) >= 0.95 * ball_reps,
      f"{len(throws)} of {ball_reps} reps")

ccuts = pd.read_parquet(FEATURES / "combine_cuts.parquet")
wc = ccuts[ccuts.drill_type == "SKILL_DRILLS_WR"].merge(throws, on="event_id", how="left")
wc["t_apex"] = wc.apex_frame * C.DT
wc["after_throw"] = wc.t_apex - wc.t_throw
timing = wc.groupby("slot").agg(
    cuts=("event_id", "size"), reps_with_ball=("t_throw", "count"),
    median_apex_s=("t_apex", "median"), median_throw_s=("t_throw", "median"),
    median_apex_after_throw_s=("after_throw", "median"),
    share_apex_after_1_5s=("t_apex", lambda s: (s >= R.MIN_APEX_AFTER_SNAP - 1e-9).mean()))
timed = timing.reps_with_ball >= MIN_BALL_REPS
timing["route_break"] = timed & (timing.median_apex_after_throw_s <= R.MAX_AFTER_THROW + 1e-9)
for twin, src in TWIN_SLOTS.items():
    timing.loc[twin, "route_break"] = timing.loc[src, "route_break"]
timing["why"] = np.select(
    [timing.route_break & ~timed, timing.route_break, ~timed],
    ["same break as " + timing.index.map(lambda s: TWIN_SLOTS.get(s, "")), "break at or before the throw",
     "no ball tracking to time it"], "after the catch")
section("Which combine slots are route breaks", timing.sort_values(["route_break", "median_apex_s"],
                                                                    ascending=[False, True]), index=True,
        note=f"A slot counts if, in reps with ball tracking (≥ {MIN_BALL_REPS}), its median apex comes no more "
             f"than {R.MAX_AFTER_THROW} s after the throw: the game rule. The throw is the first frame the ball "
             f"moves faster than {BALL_THROWN:.0f} yd/s. Slots without enough ball reps are dropped unless they are "
             "the same break logged under a second drill name. Then each cut also needs its apex ≥ "
             f"{R.MIN_APEX_AFTER_SNAP:.1f} s into the rep (the game's entry-window rule), which removes most "
             "slant breaks, as it does in games.")

keep = wc[wc.slot.map(timing.route_break) & (wc.t_apex >= R.MIN_APEX_AFTER_SNAP - 1e-9)].copy()
keep = keep[keep.groupby("slot")["event_id"].transform("size") >= F.MIN_CUTS_PER_SLOT].copy()
for m in PRIMARY:
    g = keep.groupby("slot")[m]
    keep[f"{m}_z"] = (keep[m] - g.transform("mean")) / g.transform("std")
matched = keep.groupby("nfl_id")[[f"{m}_z" for m in PRIMARY]].mean()
matched.columns = [f"match_{m}" for m in PRIMARY]
matched["match_n_cuts"] = keep.groupby("nfl_id").size()
q = q.join(matched)
check("every qualified WR keeps matched combine cuts", bool(q.match_entry_speed.notna().all()),
      f"{q.match_entry_speed.notna().sum()} of {len(q)}; median {q.match_n_cuts.median():.0f} cuts "
      f"(all drill cuts: {q.pos_n_cuts.median():.0f})")

crel_all = F.split_half_reliability(wc, n_iter=200, metrics=tuple(PRIMARY)).set_index("feature").reliability
crel_match = F.split_half_reliability(keep, n_iter=200, metrics=tuple(PRIMARY), min_drills=3
                                      ).set_index("feature").reliability
gcuts = pd.read_parquet(FEATURES / "game_cuts.parquet")
grel = split_half(gcuts, [f"{m}_z" for m in PRIMARY], players=q.index).set_index("feature").reliability
grel.index = grel.index.str.removesuffix("_z")

rows = []
for m in PRIMARY:
    for label, col in (("all drill cuts (Phase 3)", f"pos_{m}"), ("matched cuts", f"match_{m}")):
        t1 = player_test(q, f"game_{m}", col, standardize_y=True)
        t3 = player_test(q, "soe", col, standardize_y=False)
        rel = crel_all[m] if col.startswith("pos") else crel_match[m]
        rows.append({
            "signal": NAMES[m], "combine cuts": label, "combine reliability": rel,
            "r with combine (all cuts)": q[col].corr(q[f"pos_{m}"]),
            "raw r, combine → game": q[col].corr(q[f"game_{m}"]),
            "Test 1 β (SD per SD)": t1["coef"], "Test 1 95% CI": ci_str(t1["lo"], t1["hi"]), "Test 1 p": pfmt(t1["p"]),
            "Test 3 (yd SOE per SD)": t3["coef"], "Test 3 95% CI": ci_str(t3["lo"], t3["hi"], 3),
            "Test 3 p": pfmt(t3["p"]),
        })
s1 = pd.DataFrame(rows)
section("Tests 1 and 3 on matched combine cuts", s1,
        note=f"{len(q)} qualified WRs, the Phase 3 models unchanged (baseline + one combine trait; HC3 p; "
             f"{N_BOOT:,} player-bootstrap resamples). The Phase 3 rows reproduce `reports/03_models.md`. "
             "No Holm correction here: these are sensitivity checks, not new tests.")

# %% [markdown]
# ## 2. Positive control: top speed in the 40 vs peak speed on go routes

# %%
forty = con.sql("""SELECT event_id, nfl_id, time, x, y FROM combine_tracking
                   WHERE entity_type = 'PLAYER' AND drill_type = 'FORTY_YARD_DASH'
                   ORDER BY event_id, time""").df()
peaks = []
for (eid, pid), r in forty.groupby(["event_id", "nfl_id"], sort=False):
    if len(r) > 2 * C.HALF_WINDOW:
        peaks.append({"nfl_id": pid, "peak": C.kinematics(r.x.to_numpy(float), r.y.to_numpy(float))["speed"].max()})
combine_top = pd.DataFrame(peaks).groupby("nfl_id").peak.max().rename("combine_top_speed")

routes = pd.read_parquet(FEATURES / "game_routes.parquet")
go = routes[(routes.route_ran == "GO") & routes.t_snap.notna() & routes.t_pass.notna()]
go = go[go.nfl_id.isin(q.index)]
frames = R.route_frames(con, go)
route_peak = []
for key, r in frames.groupby(R.KEY, sort=False):
    if len(r) > 2 * C.HALF_WINDOW:
        route_peak.append((*key, C.kinematics(r.x.to_numpy(float), r.y.to_numpy(float))["speed"].max()))
route_peak = pd.DataFrame(route_peak, columns=[*R.KEY, "peak"])
go_n = route_peak.groupby("nfl_id").size()
game_top = route_peak[route_peak.nfl_id.isin(go_n[go_n >= MIN_GO_ROUTES].index)].groupby("nfl_id").peak.mean()
q = q.join(combine_top).join(game_top.rename("game_top_speed"))

pc = q.dropna(subset=["combine_top_speed", "game_top_speed"])
check("positive control has most qualified WRs", len(pc) >= 45, f"{len(pc)} of {len(q)}")
rel_game_top = split_half(route_peak, ["peak"], players=pc.index).reliability.iloc[0]
two = (forty.assign(n=forty.groupby("nfl_id").event_id.transform("nunique"))
       .query("n >= 2").groupby(["nfl_id", "event_id"]).size().reset_index().nfl_id.unique())
rep_peak = pd.DataFrame([{"nfl_id": pid, "event_id": eid,
                          "peak": C.kinematics(r.x.to_numpy(float), r.y.to_numpy(float))["speed"].max()}
                         for (eid, pid), r in forty[forty.nfl_id.isin(two)].groupby(["event_id", "nfl_id"])
                         if len(r) > 2 * C.HALF_WINDOW])
first_two = rep_peak.sort_values("event_id").groupby("nfl_id").head(2)
first_two["rep"] = first_two.groupby("nfl_id").cumcount()
wide = first_two.pivot(index="nfl_id", columns="rep", values="peak").dropna()
r12 = wide[0].corr(wide[1])
rel_combine_top = 2 * r12 / (1 + r12)

r, lo, hi = r_ci(pc.combine_top_speed, pc.game_top_speed)
ceiling = np.sqrt(rel_combine_top * rel_game_top)
positive_control = {"wrs": len(pc), "r": r, "boot_lo": lo, "boot_hi": hi, "ceiling": ceiling,
                    "combine_reliability": rel_combine_top, "game_reliability": rel_game_top}
beyond = player_test(pc, "game_top_speed", "combine_top_speed", standardize_y=True)
r40, lo40, hi40 = r_ci(pc.forty.fillna(pc.forty.median()), pc.game_top_speed)
es = q[["pos_entry_speed", "game_entry_speed"]].dropna()
r_es, lo_es, hi_es = r_ci(es.pos_entry_speed, es.game_entry_speed)
ctrl = pd.DataFrame([
    {"link": "40-yd top speed (tracked) → go-route peak speed", "WRs": len(pc), "raw r": r,
     "95% CI": ci_str(lo, hi), "combine reliability": rel_combine_top, "game reliability": rel_game_top,
     "ceiling √(rel·rel)": ceiling},
    {"link": "40 time (stopwatch) → go-route peak speed", "WRs": len(pc), "raw r": r40,
     "95% CI": ci_str(lo40, hi40), "combine reliability": np.nan, "game reliability": rel_game_top,
     "ceiling √(rel·rel)": np.nan},
    {"link": "speed into the break (all drill cuts) → same in games", "WRs": len(es), "raw r": r_es,
     "95% CI": ci_str(lo_es, hi_es), "combine reliability": crel_all["entry_speed"],
     "game reliability": grel["entry_speed"], "ceiling √(rel·rel)": np.sqrt(crel_all["entry_speed"] * grel["entry_speed"])},
]).set_index("link")
section("Positive control: does the pipeline find transfer where it should?", ctrl, index=True,
        note=f"Combine: the highest smoothed speed in any tracked 40 rep (same Savitzky–Golay smoothing as the cut "
             f"detector). Games: the mean, over a WR's regular-season go routes (≥ {MIN_GO_ROUTES}), of the route's "
             "peak smoothed speed from the snap to 1 s after the throw. Combine reliability: first vs second 40 rep "
             f"({len(wide)} WRs with two reps), Spearman–Brown corrected; game reliability: split-half across games. "
             f"Beyond the baseline (which already holds the 40 time), tracked top speed adds β = {beyond['coef']:+.2f} "
             f"SD per SD {ci_str(beyond['lo'], beyond['hi'])}, p = {pfmt(beyond['p'])}.")

# %% [markdown]
# ## 3. Play context: does Test 2's within-receiver effect survive press, alignment, situation and coverage?

# %%
SIGNALS = PRIMARY + ["peak_decel"]
NAMES["peak_decel"] = "peak braking (secondary)"
KEY = R.KEY

troutes = routes[routes.nfl_id.isin(q.index) & routes.soe.notna()]
pre_cuts = gcuts[gcuts.nfl_id.isin(q.index) & (gcuts.t_after_throw <= 1e-9)]
agg = {f"{m}_z": (f"{m}_z", "mean") for m in SIGNALS}
t2 = troutes[[*KEY, "soe", "route_ran", "coverage"]].merge(
    pre_cuts.groupby(KEY).agg(**agg, break_depth=("depth", "mean"), break_turn=("turn_deg", "mean")).reset_index(),
    on=KEY)
check("Test 2 sample is Phase 3's 8,371 routes", len(t2) == 8371, f"{len(t2):,}")

con.register("t2_keys", t2[KEY])
ctx = con.sql("""
    SELECT pp.game_id, pp.play_id, pp.nfl_id, pp.cushion, pp.lined_up_position, pp.receiver_alignment,
           pp.offense_formation, pp.team_coverage_type, pp.down, pp.yards_to_go, pp.yardline_side,
           pp.yardline_number, pp.possession_team, pp.quarter, CAST(pp.game_clock AS VARCHAR) AS game_clock,
           pp.pre_snap_home_team_win_probability AS home_wp, pp.in_motion_at_ball_snap AS in_motion,
           g.home_team_abbr, g.visitor_team_abbr
    FROM player_play_reg pp JOIN games g USING (game_id) JOIN t2_keys USING (game_id, play_id, nfl_id)""").df()
con.unregister("t2_keys")
t2 = t2.merge(ctx, on=KEY, how="left")
check("every Test 2 route has its play context", bool(t2.down.notna().all()), f"{t2.down.notna().sum():,}")
off_home = t2.possession_team == t2.home_team_abbr
check("the offense is always the home or visiting team",
      bool((off_home | (t2.possession_team == t2.visitor_team_abbr)).all()))

clock = t2.game_clock.str.split(":", expand=True).astype(int)  # mm:ss:00
check("game clock reads as minutes:seconds", bool((clock[0] <= 15).all() & (clock[1] < 60).all()))
secs_left = clock[0] * 60 + clock[1]
to_goal = pd.Series(np.where(t2.yardline_side.isna(), 50, np.where(
    t2.yardline_side == t2.possession_team, 100 - t2.yardline_number, t2.yardline_number)), index=t2.index)
off_wp = pd.Series(np.where(off_home, t2.home_wp, 1 - t2.home_wp), index=t2.index)


def bins(x: pd.Series, edges, labels) -> pd.Series:
    return pd.cut(x.astype(float), edges, labels=labels, right=False).astype(str)


CUSHION_LABELS = ["< 3 yd (press)", "3–6 yd", "6–9 yd", "9+ yd"]
t2["c_cushion"] = bins(t2.cushion, [0, 3, 6, 9, np.inf], CUSHION_LABELS).where(t2.cushion.notna(), "missing")
t2["c_align"] = t2.lined_up_position.map({"WR": "outside", "SLOT_WR": "slot"}).fillna("other")
t2["c_rec_align"] = t2.receiver_alignment.where(t2.receiver_alignment.isin(["2x2", "3x1", "3x2"]), "other")
t2["c_formation"] = t2.offense_formation.where(t2.offense_formation.isin(["SHOTGUN", "SINGLEBACK", "EMPTY"]), "other")
t2["c_motion"] = t2.in_motion.astype(str)
t2["c_to_goal"] = bins(to_goal, [0, 11, 21, 41, 81, 101], ["1–10", "11–20", "21–40", "41–80", "81–99"])
t2["c_down"] = t2.down.astype(int).clip(upper=4).astype(str)
t2["c_to_go"] = bins(t2.yards_to_go, [0, 4, 7, 11, np.inf], ["1–3", "4–6", "7–10", "11+"])
t2["c_two_min"] = (t2.quarter.isin([2, 4]) & (secs_left <= 120)).astype(str)
t2["c_wp"] = bins(off_wp, [0, 0.1, 0.5, 0.9, 1.01], ["< 10%", "10–50%", "50–90%", "90%+"])
t2["c_quarter"] = t2.quarter.astype(int).clip(upper=4).astype(str)
SHELLS = ["COVER_0_MAN", "COVER_1_MAN", "COVER_2_MAN", "COVER_2_ZONE", "COVER_3_ZONE", "COVER_4_ZONE", "COVER_6_ZONE"]
t2["c_coverage"] = t2.team_coverage_type.where(t2.team_coverage_type.isin(SHELLS), "other")
t2["c_route"] = t2.route_ran
t2["break_depth2"], t2["break_turn2"] = t2.break_depth ** 2, t2.break_turn ** 2

GROUPS = {
    "press: cushion at the snap": ["c_cushion"],
    "alignment, formation, motion": ["c_align", "c_rec_align", "c_formation", "c_motion"],
    "field position, down, distance, game state": ["c_to_goal", "c_down", "c_to_go", "c_two_min", "c_wp", "c_quarter"],
    "coverage shell (Cover 0–6)": ["c_coverage"],
    "route design: route type, break depth and angle": ["c_route", "break_depth", "break_depth2", "break_turn",
                                                         "break_turn2"],
}
ALL = [c for cols in GROUPS.values() for c in cols]


def design(d: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    cat = [c for c in cols if c.startswith("c_")]
    parts = [pd.get_dummies(d[cat], drop_first=True, dtype=float)] if cat else []
    parts += [d[[c for c in cols if not c.startswith("c_")]].astype(float)]
    return pd.concat(parts, axis=1)


def within(D: pd.DataFrame | pd.Series, d: pd.DataFrame, by: list[str]):
    """Deviation from the mean of each fixed-effect group (a WR, a WR in one game, a WR against man...)."""
    return D - D.groupby([d[c] for c in by]).transform("mean")


def cluster_fit(d: pd.DataFrame, D: pd.DataFrame, y: np.ndarray, report_cols: list[str]) -> dict[str, dict]:
    """OLS of y on D (both already within-transformed): coefficient, player-clustered CR1 p and player-cluster
    bootstrap CI for each of `report_cols`. Bootstrap fits sum per-player cross-products, so they are cheap."""
    fit = M.ols(y, D.to_numpy(), cluster=d.nfl_id.to_numpy())
    X1 = np.column_stack([np.ones(len(D)), D.to_numpy()])
    players = d.nfl_id.unique()
    codes = pd.Categorical(d.nfl_id, categories=players).codes
    XtX = np.stack([X1[codes == i].T @ X1[codes == i] for i in range(len(players))])
    Xty = np.stack([X1[codes == i].T @ y[codes == i] for i in range(len(players))])
    out = {}
    for c in report_cols:
        j = list(D.columns).index(c) + 1
        draws = M.bootstrap(lambda idx: np.linalg.lstsq(XtX[idx].sum(0), Xty[idx].sum(0), rcond=None)[0][j],
                            players, N_BOOT, seed=0)
        lo, hi = M.percentile_ci(draws)
        out[c] = {"b": fit["coef"][j], "lo": lo, "hi": hi, "p": fit["p"][j]}
    return out


def fe_effect(m: str, cols: list[str], by: list[str]) -> dict:
    """SOE on the route's break score plus controls, with fixed effects for `by`; the break score's coefficient.

    With no controls and WR fixed effects this is exactly Test 2's within-player effect. Break scores are in
    route-level SD units, as in Phase 3. Player-clustered CR1 p; player-cluster bootstrap CI.
    """
    x = (t2[f"{m}_z"] / t2[f"{m}_z"].std()).rename("x")
    D = within(pd.concat([x, design(t2, cols)], axis=1), t2, by)
    out = cluster_fit(t2, D, within(t2.soe, t2, by).to_numpy(), ["x"])["x"]
    zd = D.drop(columns="x").to_numpy()
    out["r2_context"] = 0.0
    if zd.shape[1]:
        xd = D.x.to_numpy()
        res = xd - zd @ np.linalg.lstsq(zd, xd, rcond=None)[0]
        out["r2_context"] = 1 - (res @ res) / (xd @ xd)
    return out


rows = [("none (Phase 3 Test 2)", [], ["nfl_id"])]
rows += [(f"+ {name}", cols, ["nfl_id"]) for name, cols in GROUPS.items()]
rows += [("+ all of the above", ALL, ["nfl_id"]),
         ("+ all of the above, routes in the same game only", ALL, ["nfl_id", "game_id"])]
fe = {(label, m): fe_effect(m, cols, by) for label, cols, by in rows for m in SIGNALS}
base = {m: fe[("none (Phase 3 Test 2)", m)]["b"] for m in SIGNALS}
check("no-control row reproduces Phase 3 Test 2 (entry speed +0.110, retention +0.078)",
      abs(base["entry_speed"] - 0.110) < 0.0015 and abs(base["speed_retention"] - 0.078) < 0.0015,
      f"{base['entry_speed']:+.4f}, {base['speed_retention']:+.4f}")

ctx_tab = pd.DataFrame(
    {NAMES[m]: [f"{fe[(label, m)]['b']:+.3f} {ci_str(fe[(label, m)]['lo'], fe[(label, m)]['hi'], 3)}, "
                f"p {pfmt(fe[(label, m)]['p'])}" for label, _, _ in rows] for m in SIGNALS},
    index=[label for label, _, _ in rows]).rename_axis("controls added to Test 2")
same_game = t2.groupby(["nfl_id", "game_id"]).soe.transform("size") > 1
section("Play context: Test 2 with controls", ctx_tab, index=True,
        note="Within-receiver effect: yd of separation over expected per SD of the route's break score, comparing a "
             "WR's routes with each other (WR fixed effects; with no controls this is exactly Phase 3's Test 2). "
             f"{len(t2):,} routes, {t2.nfl_id.nunique()} WRs; player-clustered p, {N_BOOT:,} player-bootstrap "
             "resamples. Every control except break depth and angle is fixed before the snap. Press = cushion "
             "(defender distance at the snap) under 3 yd. The last row compares only routes a WR ran in the same "
             f"game (same QB, opponent, weather): {same_game.sum():,} routes in "
             f"{t2[same_game].groupby(['nfl_id', 'game_id']).ngroups:,} WR-games share a game with another.")

r2_tab = pd.DataFrame({"share of within-WR break-score variance explained by all controls":
                       [fe[("+ all of the above", m)]["r2_context"] for m in SIGNALS]},
                      index=[NAMES[m] for m in SIGNALS]).rename_axis("signal")
section("How much of the break score is play context?", r2_tab, index=True,
        note="R² of each within-WR break score on every control above (within-WR deviations). Small values mean "
             "context barely moves the break score, so it can't confound much.")

dev_es = within(t2.entry_speed_z / t2.entry_speed_z.std(), t2, ["nfl_id"])
dev_soe = within(t2.soe, t2, ["nfl_id"])
cush = pd.DataFrame({"routes": t2.groupby("c_cushion").size(),
                     "share": t2.groupby("c_cushion").size() / len(t2),
                     "speed into the break vs WR's own average (SD)": dev_es.groupby(t2.c_cushion).mean(),
                     "SOE vs WR's own average (yd)": dev_soe.groupby(t2.c_cushion).mean()}
                    ).reindex([*CUSHION_LABELS, "missing"]).rename_axis("cushion at the snap")
section("Press vs off coverage, within receivers", cush, index=True,
        note="Mean deviation from each WR's own average, by cushion. If press both slowed the break and closed "
             "separation, this is where it would show.")

# %% [markdown]
# ### 3b. Exploratory: is "heavier WRs are less open" an alignment effect?

# %%
qroutes = routes[routes.nfl_id.isin(q.index) & routes.soe.notna()]
con.register("q_keys", qroutes[KEY])
align = con.sql("""SELECT game_id, play_id, nfl_id, lined_up_position FROM player_play_reg
                   JOIN q_keys USING (game_id, play_id, nfl_id)""").df()
con.unregister("q_keys")
qa = qroutes.merge(align, on=KEY, how="left")
q["slot_share"] = qa.assign(slot=qa.lined_up_position == "SLOT_WR").groupby("nfl_id").slot.mean()
soe_by_align = qa.groupby(qa.lined_up_position.map({"WR": "outside", "SLOT_WR": "slot"}).fillna("other")).soe.agg(
    ["size", "mean"])
B, _ = M.baseline_matrix(q)
f0 = M.ols(q.soe.to_numpy(), M.zscore(B).to_numpy())
B1 = B.assign(slot_share=q.slot_share)
f1 = M.ols(q.soe.to_numpy(), M.zscore(B1).to_numpy())
w = list(B.columns).index("weight") + 1
wt = pd.DataFrame([
    {"model": "baseline (Phase 3 exploratory)", "weight (yd SOE per SD)": f0["coef"][w], "weight p": pfmt(f0["p"][w]),
     "slot share (yd SOE per SD)": np.nan, "slot share p": ""},
    {"model": "baseline + share of routes from the slot", "weight (yd SOE per SD)": f1["coef"][w],
     "weight p": pfmt(f1["p"][w]), "slot share (yd SOE per SD)": f1["coef"][-1], "slot share p": pfmt(f1["p"][-1])},
]).set_index("model")
section("Exploratory: weight, alignment and separation", wt, index=True,
        note=f"{len(q)} qualified WRs; HC3 p. Weight and slot share correlate r = "
             f"{q.combine_weight.corr(q.slot_share):+.2f}. Route-level SOE by alignment: "
             + ", ".join(f"{k} {v['mean']:+.2f} yd ({int(v['size']):,} routes)" for k, v in soe_by_align.iterrows())
             + ". SOE adjusts for route, coverage and time to throw, not alignment.")

# %% [markdown]
# ## 4. Man vs zone: where does the break buy separation?

# %%
cov = t2[t2.coverage.isin(["MAN_COVERAGE", "ZONE_COVERAGE"])].copy()
cov["is_man"] = (cov.coverage == "MAN_COVERAGE").astype(float)
check("almost every Test 2 route has a man / zone label", len(cov) >= 0.99 * len(t2), f"{len(cov):,} of {len(t2):,}")
BY_COV = ["nfl_id", "coverage"]


def split_effect(m: str, cols: list[str]) -> dict:
    """Within-WR effect of the break score on SOE, separately against man and zone, and the man − zone gap.

    Fixed effects for each WR x coverage: a WR's man routes are compared with their own man routes, zone with
    zone. Break scores are in Test 2's route-level SD units.
    """
    x = cov[f"{m}_z"] / t2[f"{m}_z"].std()
    y = within(cov.soe, cov, BY_COV).to_numpy()
    Z = design(cov, cols)
    slopes = pd.DataFrame({"man": x * cov.is_man, "zone": x * (1 - cov.is_man)})
    gap = pd.DataFrame({"all": x, "man_minus_zone": x * cov.is_man})
    out = cluster_fit(cov, within(pd.concat([slopes, Z], axis=1), cov, BY_COV), y, ["man", "zone"])
    out["gap"] = cluster_fit(cov, within(pd.concat([gap, Z], axis=1), cov, BY_COV), y,
                             ["man_minus_zone"])["man_minus_zone"]
    return out


def est_ci(r: dict) -> str:
    return f"{r['b']:+.3f} {ci_str(r['lo'], r['hi'], 3)}"


mz_rows = []
for m in SIGNALS:
    for label, cols in (("none", []), ("all play-context controls", ALL)):
        e = split_effect(m, cols)
        mz_rows.append({"signal": NAMES[m], "controls": label, "vs man (yd per SD)": est_ci(e["man"]),
                        "vs man p": pfmt(e["man"]["p"]), "vs zone (yd per SD)": est_ci(e["zone"]),
                        "vs zone p": pfmt(e["zone"]["p"]), "man − zone": est_ci(e["gap"]),
                        "man − zone p": pfmt(e["gap"]["p"])})
n_man = int(cov.is_man.sum())
section("Man vs zone: Test 2 by coverage", pd.DataFrame(mz_rows),
        note=f"{len(cov):,} Test 2 routes with a coverage label ({n_man:,} against man, {len(cov) - n_man:,} against "
             f"zone), {cov.nfl_id.nunique()} WRs. Fixed effects for each WR × coverage, so a WR's man routes are "
             "compared with their own man routes and zone with zone; yd of SOE per SD of the break score (Test 2's "
             f"units). Player-clustered p, {N_BOOT:,} player-bootstrap resamples. Exploratory and uncorrected. "
             "Prediction stated before computing: speed into the break matters more against man.")

fifths = []
for m in PRIMARY:
    xw = within(cov[f"{m}_z"] / t2[f"{m}_z"].std(), cov, BY_COV)
    yw = within(cov.soe, cov, BY_COV)
    for c, label in (("MAN_COVERAGE", "man"), ("ZONE_COVERAGE", "zone")):
        k = cov.coverage == c
        est = yw[k].groupby(pd.qcut(xw[k], 5, labels=False)).mean()
        fifths.append({"signal": NAMES[m], "coverage": label, "routes": int(k.sum()),
                       "SOE, top fifth vs bottom fifth of the WR's own routes (yd)": est.iloc[-1] - est.iloc[0]})
section("Man vs zone in plain terms", pd.DataFrame(fifths),
        note="Each WR's routes against one coverage, split into fifths by break score relative to the WR's own "
             "average against that coverage (as in the writeup's Figure 3, there pooled over coverages).")

# %% [markdown]
# ## 5. What a yard of separation is worth

# %%
s5 = routes[routes.soe.notna() & routes.coverage.isin(["MAN_COVERAGE", "ZONE_COVERAGE"])].copy()
con.register("s5_keys", s5[KEY])
s5 = s5.merge(con.sql("""SELECT game_id, play_id, nfl_id, pass_length, down, yards_to_go FROM player_play_reg
                         JOIN s5_keys USING (game_id, play_id, nfl_id)""").df(), on=KEY, how="left")
con.unregister("s5_keys")
s5["targeted"] = s5.target.fillna(False).astype(bool)
s5["c_route"], s5["c_cov"], s5["c_ttt"] = s5.route_ran, s5.coverage, s5.ttt_bin
s5["c_down"] = s5.down.astype(int).clip(upper=4).astype(str)
s5["c_to_go"] = bins(s5.yards_to_go, [0, 4, 7, 11, np.inf], ["1–3", "4–6", "7–10", "11+"])
s5["c_air"] = bins(s5.pass_length, [-np.inf, 0, 5, 10, 15, 20, 30, np.inf],
                   ["< 0", "0–4", "5–9", "10–14", "15–19", "20–29", "30+"])
tg = s5[s5.targeted].copy()
tg["complete"] = (tg.pass_result == "C").astype(float)
check("every target is a completion, incompletion or interception", bool(tg.pass_result.isin(["C", "I", "IN"]).all()),
      f"{len(tg):,} targets")

SEP_BINS, SEP_LABELS = [0, 1, 2, 3, 4, 6, np.inf], ["< 1 yd", "1–2", "2–3", "3–4", "4–6", "6+"]
raw = pd.DataFrame({
    "routes": s5.groupby(bins(s5.separation, SEP_BINS, SEP_LABELS)).size(),
    "targeted %": 100 * s5.groupby(bins(s5.separation, SEP_BINS, SEP_LABELS)).targeted.mean(),
    "targets": tg.groupby(bins(tg.separation, SEP_BINS, SEP_LABELS)).size(),
    "completion %": 100 * tg.groupby(bins(tg.separation, SEP_BINS, SEP_LABELS)).complete.mean(),
    "EPA per target": tg.groupby(bins(tg.separation, SEP_BINS, SEP_LABELS)).expected_points_added.mean(),
}).reindex(SEP_LABELS).rename_axis("separation at the throw")
section("Separation and outcomes (raw)", raw, index=True,
        note=f"{len(s5):,} WR routes with separation at the throw ({s5.nfl_id.nunique()} WRs, screens excluded), "
             f"{len(tg):,} of them targeted. Median separation on targets: "
             f"{tg[tg.coverage == 'MAN_COVERAGE'].separation.median():.1f} yd against man, "
             f"{tg[tg.coverage == 'ZONE_COVERAGE'].separation.median():.1f} yd against zone.")

CONTROLS = ["c_route", "c_ttt", "c_down", "c_to_go"]
OUTCOMES = {  # label: (frame, outcome, scale, controls)
    "completion % if targeted": (tg, "complete", 100, CONTROLS + ["c_air"]),
    "EPA per target": (tg, "expected_points_added", 1, CONTROLS + ["c_air"]),
    "chance of being targeted (%)": (s5, "targeted", 100, CONTROLS),
}


def per_yard(d: pd.DataFrame, y: str, scale: float, cols: list[str]) -> dict:
    """Effect of one yard of separation on y: pooled (WR fixed effects, coverage as a control) and against man and
    zone separately (WR x coverage fixed effects, as in section 4). Linear in separation within each coverage."""
    yy = d[y].astype(float) * scale
    pooled = cluster_fit(d, within(pd.concat([d.separation.rename("sep"), design(d, cols + ["c_cov"])], axis=1),
                                   d, ["nfl_id"]), within(yy, d, ["nfl_id"]).to_numpy(), ["sep"])["sep"]
    man = (d.coverage == "MAN_COVERAGE").astype(float)
    X = pd.DataFrame({"man": d.separation * man, "zone": d.separation * (1 - man)})
    split = cluster_fit(d, within(pd.concat([X, design(d, cols)], axis=1), d, BY_COV),
                        within(yy, d, BY_COV).to_numpy(), ["man", "zone"])
    return {"all": pooled, **split}


slopes = {label: per_yard(*spec) for label, spec in OUTCOMES.items()}
fmt1 = {"completion % if targeted": 1, "EPA per target": 3, "chance of being targeted (%)": 1}
slope_tab = pd.DataFrame(
    {col: [f"{slopes[o][k]['b']:+.{fmt1[o]}f} {ci_str(slopes[o][k]['lo'], slopes[o][k]['hi'], fmt1[o])}, "
           f"p {pfmt(slopes[o][k]['p'])}" for o in OUTCOMES]
     for col, k in (("all coverages", "all"), ("vs man", "man"), ("vs zone", "zone"))},
    index=list(OUTCOMES)).rename_axis("per yard of separation at the throw")
section("What one yard of separation is worth", slope_tab, index=True,
        note="Within-WR linear-probability / OLS models, so a WR's own hands and role cancel out. Controls: route "
             "type, time-to-throw bin, down, yards to go, and for targets the pass's air yards (7 bins). Player-"
             f"clustered p, {N_BOOT:,} player-bootstrap resamples. Completion % rises steeply at tight separation "
             "(table above), which is where man coverage lives, so a yard is worth more against man.")

fifth = pd.DataFrame(fifths).set_index(["signal", "coverage"])["SOE, top fifth vs bottom fifth of the WR's own routes (yd)"]
same_game = ("+ all of the above, routes in the same game only", "entry_speed")
effects = [  # label, yd of separation, which slope applies
    ("speed into the break, +1 SD (Phase 3 Test 2)", base["entry_speed"], "all"),
    ("speed into the break, +1 SD (all play context, same game)", fe[same_game]["b"], "all"),
    ("vs man: best vs worst fifth of a WR's breaks by speed into the break",
     fifth[(NAMES["entry_speed"], "man")], "man"),
    ("vs zone: best vs worst fifth by speed into the break", fifth[(NAMES["entry_speed"], "zone")], "zone"),
    ("vs man: best vs worst fifth by peak lateral acceleration",
     fifth[(NAMES["peak_lat_accel"], "man")], "man"),
]
trans = pd.DataFrame([{"effect": label, "yd of separation": yd,
                       **{o: yd * slopes[o][k]["b"] for o in OUTCOMES}} for label, yd, k in effects]).set_index("effect")
section("The break effects in football units", trans, index=True,
        note="Yards of separation from sections 3–4 times the per-yard values above (the man or zone value for "
             "man- or zone-only effects). Point estimates, to give the size in familiar units; both pieces carry "
             "their own uncertainty. Completion % and EPA apply when the WR is targeted.")

# %%
(FEATURES / "sensitivity.json").write_text(json.dumps({"positive_control": positive_control}, indent=1),
                                           encoding="utf-8")
matched.to_parquet(FEATURES / "combine_matched_features.parquet")  # for the F5 scouting card in 04_writeup.py
report[6:6] = ["## Checks", "", *checks, ""]
out = ROOT / "reports" / "03b_sensitivity.md"
out.write_text("\n".join(report) + "\n", encoding="utf-8")
print(f"\n{len(checks)} checks passed; wrote {out.relative_to(ROOT)}")

In [ ]:
%%writefile notebooks/04_writeup.py
# %% [markdown]
# # Phase 4 — writeup figures and numbers
#
# Builds the figures for `writeup/writeup.md` (PLAN.md §8) and `reports/04_numbers.md`, which holds every number
# the writeup cites that isn't already in reports 01–03. Descriptive only: no new hypothesis tests.
#
# - F1 `04_signature.png`: one WR's combine speed-out vs the same WR's game out-breaks (path and speed)
# - F2 `04_within.png`: separation vs break score within receivers, binned, against man and against zone
# - F4 `04_ceiling.png`: how high an observed combine–game r could get vs what was observed, with the
#   positive control (top speed, from `03b_sensitivity.py`)
# - F5 `04_card.png`: in-season scouting card for the F1 WR
# - F6 `04_stabilize.png`: how many routes before game break metrics are trustworthy
# - `04_cover.png` (560 × 120) and `04_cover@2x.png`: the Kaggle card image, F1's paths turned sideways
# (F3 is `03_tests.png` from Phase 3; T1 is a table in the writeup.)
#
# Run: `python notebooks/04_writeup.py` (after notebooks 01–03 and 03b)

# %%
import json
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))

import matplotlib  # noqa: E402

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
from matplotlib.lines import Line2D  # noqa: E402

from src import cuts as C  # noqa: E402
from src import features as F  # noqa: E402
from src import models as M  # noqa: E402
from src import routes as R  # noqa: E402
from src.data import connect  # noqa: E402
from src.stats import split_half  # noqa: E402

FEATURES = ROOT / "data" / "features"
FIGURES = ROOT / "reports" / "figures"
MIN_ROUTES = 100
KEY = R.KEY
PRIMARY = ["entry_speed", "speed_retention", "peak_lat_accel"]
NAMES = {"entry_speed": "speed into the break", "speed_retention": "speed retention",
         "peak_lat_accel": "peak lateral acceleration"}

numbers: list[str] = ["# Phase 4 — numbers cited in the writeup", "",
                      "Generated by `notebooks/04_writeup.py`. Do not edit by hand.", ""]
checks: list[str] = []


def check(name: str, ok: bool, detail: str = "") -> None:
    checks.append(f"- [{'x' if ok else ' '}] {name}" + (f" — {detail}" if detail else ""))
    assert ok, f"{name}: {detail}"


def note(label: str, value) -> None:
    numbers.append(f"- {label}: **{value}**")
    print(f"{label}: {value}")


SERIES = ["#2a78d6", "#eb6834", "#1baf7a"]
SURFACE, INK, INK2, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"
GAME, COMBINE = SERIES[0], SERIES[1]
# Man vs zone in F2: slots 1 and 3, which validate as a pair; orange stays "combine" across every figure.
MAN, ZONE = SERIES[0], SERIES[2]
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "axes.titlecolor": INK, "axes.titlesize": 12, "axes.labelsize": 11, "xtick.labelsize": 10, "ytick.labelsize": 10,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.6, "lines.linewidth": 2, "font.size": 11,
})
# Kaggle shows figures at its column width (~60% of the wide ones), so text is sized for that, saved at a sharp dpi.
DPI, SMALL, SUPTITLE = 150, 10, 13

con = connect()
wr = pd.read_parquet(FEATURES / "wr_analysis.parquet")
q = wr[wr.routes.fillna(0) >= MIN_ROUTES]
gcuts = pd.read_parquet(FEATURES / "game_cuts.parquet")
ccuts = pd.read_parquet(FEATURES / "combine_cuts.parquet")
groutes = pd.read_parquet(FEATURES / "game_routes.parquet")
if not (FEATURES / "sensitivity.json").exists():
    raise SystemExit("Missing sensitivity.json: run notebooks/03b_sensitivity.py first.")
positive_control = json.loads((FEATURES / "sensitivity.json").read_text(encoding="utf-8"))["positive_control"]
check("62 qualified WRs", len(q) == 62, str(len(q)))

# %% [markdown]
# ## F1. Signature: the same break, measured twice

# %%
SPEED_OUT = ["SPEED_OUT_ROUTE_LEFT", "SPEED_OUT_LEFT"]
so = ccuts[ccuts.drill_name.isin(SPEED_OUT) & ccuts.slot.str.endswith("|L1")]
outs = gcuts[(gcuts.slot == "OUT|out") & gcuts.nfl_id.isin(q.index) & gcuts.nfl_id.isin(so.nfl_id)]
pid = outs.groupby("nfl_id").size().idxmax()   # selection rule: most game out-breaks among WRs who ran the drill
name = wr.loc[pid, "display_name"]
p_outs = outs[outs.nfl_id == pid]
p_so = so[so.nfl_id == pid]
note("F1 receiver (most game out-breaks among qualified WRs who ran the combine speed-out)",
     f"{name}, {len(p_outs)} game out-breaks, {len(p_so)} combine speed-out rep(s)")

# Speed into the break (peak speed in the 1.5 s before the apex), in raw yd/s, against the other qualified WRs.
drill_peers = so[so.nfl_id.isin(q.index)].groupby("nfl_id").entry_speed.mean()
game_peers = outs.groupby("nfl_id").entry_speed.median()
game_peers_z = outs.groupby("nfl_id").entry_speed_z.mean()   # adjusted for depth and angle, as in the analysis
c_entry, g_entry = drill_peers[pid], game_peers[pid]
note("F1 speed into the break (yd/s): combine speed-out / median game out-break",
     f"{c_entry:.2f} / {g_entry:.2f}")
note("F1 qualified WRs' median speed into the break (yd/s): drill / game out-breaks",
     f"{drill_peers.median():.2f} ({len(drill_peers)} WRs) / {game_peers.median():.2f} ({len(game_peers)} WRs)")
note("F1 receiver's percentile: drill / game out-breaks raw / game out-breaks adjusted for depth and angle",
     f"{(drill_peers < c_entry).mean() * 100:.0f} / {(game_peers < g_entry).mean() * 100:.0f} / "
     f"{(game_peers_z < game_peers_z[pid]).mean() * 100:.0f}")

cframes = con.sql(f"""SELECT event_id, time, x, y FROM combine_tracking
                     WHERE entity_type = 'PLAYER' AND event_id IN ({', '.join(f"'{e}'" for e in p_so.event_id)})
                     ORDER BY event_id, time""").df()
proutes = R.wr_routes(con)
proutes = proutes[proutes.nfl_id == pid].merge(p_outs[KEY].drop_duplicates(), on=KEY)
gframes = R.route_frames(con, proutes)
WIN = (-15, 10)  # frames around the apex: 1.5 s in, 1.0 s out


def trace(x, y, apex, mirror_to_right=True):
    """Path and speed from WIN[0] to WIN[1] frames around the apex, rotated so the approach points up.

    Frames past the end of the tracked window (it stops 1 s after the throw) are NaN, so late breaks
    stay in the figure rather than biasing it toward later throws.
    """
    k = C.kinematics(x, y)
    lo, hi = apex + WIN[0], apex + WIN[1]
    if lo < 0:
        return None
    end = min(hi, len(x) - 1)
    vx, vy = k["vx"][apex - 5:apex].mean(), k["vy"][apex - 5:apex].mean()
    ang = np.arctan2(vx, vy)                     # rotate the approach direction onto +y
    px, py = x[lo:end + 1] - x[apex], y[lo:end + 1] - y[apex]
    rx = px * np.cos(ang) - py * np.sin(ang)
    ry = px * np.sin(ang) + py * np.cos(ang)
    if mirror_to_right and rx[-1] < 0:
        rx = -rx
    pad = np.full(hi - end, np.nan)
    t = np.arange(WIN[0], WIN[1] + 1) * C.DT
    return t, np.r_[rx, pad], np.r_[ry, pad], np.r_[k["speed"][lo:end + 1], pad]


game_tr = []
for _, r in p_outs.iterrows():
    f = gframes[(gframes.game_id == r.game_id) & (gframes.play_id == r.play_id)].sort_values("time")
    tr = trace(f.x.to_numpy(float), f.y.to_numpy(float), int(r.apex_frame))
    if tr is not None:
        game_tr.append(tr)
comb_tr = []
for _, r in p_so.iterrows():
    f = cframes[cframes.event_id == r.event_id]
    tr = trace(f.x.to_numpy(float), f.y.to_numpy(float), int(r.apex_frame))
    if tr is not None:
        comb_tr.append(tr)
check("F1 has the combine rep and every game out-break traced", len(comb_tr) >= 1 and len(game_tr) == len(p_outs),
      f"{len(comb_tr)} combine, {len(game_tr)} of {len(p_outs)} game")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.6), gridspec_kw={"width_ratios": [1, 1.6]})
for t, rx, ry, s in game_tr:
    a1.plot(rx, ry, color=GAME, lw=0.8, alpha=0.18)
    a2.plot(t, s, color=GAME, lw=0.8, alpha=0.18)
S = np.vstack([s for *_, s in game_tr])
med = np.where(np.isfinite(S).mean(axis=0) >= 0.5, np.nanmedian(S, axis=0), np.nan)  # median where ≥ half have data
a2.plot(game_tr[0][0], med, color=GAME, lw=2.5, label=f"games: median of {len(game_tr)} out-breaks")
for t, rx, ry, s in comb_tr:
    a1.plot(rx, ry, color=COMBINE, lw=2.5)
    a2.plot(t, s, color=COMBINE, lw=2.5, label="combine: speed-out drill")
a2.axvline(0, color=INK2, lw=1, zorder=0)
# The qualified WRs' median speed into the break, drawn over the entry window where it is measured.
for v, col, label in ((drill_peers.median(), COMBINE, "drill"), (game_peers.median(), GAME, "game out-breaks")):
    a2.hlines(v, WIN[0] * C.DT, 0, colors=col, linestyles=(0, (1, 1.5)), lw=2, zorder=5,
              label=f"{len(drill_peers)} WRs' median, {label}: {v:.1f}")
a1.plot(0, 0, "o", ms=8, mfc=SURFACE, mec=INK, mew=1.5, zorder=4)
a1.set_aspect("equal", "datalim")
a1.set_title("Path, approach pointing up,\napex at the circle", loc="left")
a1.set_xlabel("yd")
a1.set_ylabel("yd")
a2.set_title(f"Speed into the break (peak in the 1.5 s before the apex):\n"
             f"{c_entry:.1f} yd/s in the drill, {g_entry:.1f} in games (median)", loc="left")
a2.set_xlabel("seconds from the apex of the break")
a2.set_ylabel("speed (yd/s)")
a2.legend(frameon=False, loc="lower left", fontsize=SMALL, labelcolor=INK)
fig.suptitle(f"{name}: the same speed into the out-break in the drill and in games, ranked very differently",
             x=0.01, ha="left", color=INK, fontsize=SUPTITLE)
fig.tight_layout(rect=(0, 0, 1, 0.94))
fig.savefig(FIGURES / "04_signature.png", dpi=DPI)
plt.close(fig)

# Kaggle card image (560 × 120, plus a 2× copy): F1's paths turned sideways, approach running left to right.
fig = plt.figure(figsize=(5.6, 1.2))
ax = fig.add_axes((0.47, 0.04, 0.52, 0.92))
for _, rx, ry, _ in game_tr:
    ax.plot(ry, rx, color=GAME, lw=0.6, alpha=0.22)
for _, rx, ry, _ in comb_tr:
    ax.plot(ry, rx, color=COMBINE, lw=2.2)
ax.plot(0, 0, "o", ms=5, mfc=SURFACE, mec=INK, mew=1.2, zorder=4)
ax.set_aspect("equal", "datalim")
ax.axis("off")
fig.text(0.03, 0.62, "The cut that doesn't travel", fontsize=14, color=INK, va="center")
fig.text(0.03, 0.30, "Combine drill (orange) vs NFL game breaks (blue)", fontsize=7.5, color=INK2, va="center")
for scale, suffix in ((1, ""), (2, "@2x")):
    fig.savefig(FIGURES / f"04_cover{suffix}.png", dpi=100 * scale)
plt.close(fig)

cs = comb_tr[0][3]
note("F1 combine speed-out: speed 1.5 s before / at the apex (yd/s)", f"{cs[0]:.1f} / {cs[-WIN[0]]:.1f}")
note("F1 games: median speed 1.5 s before / at the apex (yd/s)", f"{med[0]:.1f} / {med[-WIN[0]]:.1f}")

# %% [markdown]
# ## F2. Within receivers: the break buys separation against man coverage
#
# The man / zone split is from `03b_sensitivity.py` (section 4), added after the pre-registered tests.

# %%
COVERAGES = [("MAN_COVERAGE", "man", MAN, "o", -0.04), ("ZONE_COVERAGE", "zone", ZONE, "D", 0.04)]
pre = gcuts[gcuts.nfl_id.isin(q.index) & (gcuts.t_after_throw <= 1e-9)]
rt = groutes[groutes.nfl_id.isin(q.index) & groutes.soe.notna()][[*KEY, "soe", "coverage"]].merge(
    pre.groupby(KEY)[[f"{m}_z" for m in PRIMARY]].mean().reset_index(), on=KEY)
check("F2 uses the Test 2 routes (8,371)", len(rt) == 8371, f"{len(rt):,}")
rc = rt[rt.coverage.isin([c[0] for c in COVERAGES])]
fig, axes = plt.subplots(1, 3, figsize=(12, 4.2), sharey=True)
rng = np.random.default_rng(0)
for ax, m in zip(axes, PRIMARY):
    x = rc[f"{m}_z"] / rt[f"{m}_z"].std()  # Test 2's units: SD of the route-level break score
    fe = [rc.nfl_id, rc.coverage]          # a WR's man routes vs their own man routes, zone vs zone
    xw = x - x.groupby(fe).transform("mean")
    yw = rc.soe - rc.soe.groupby(fe).transform("mean")
    slopes = []
    for cov, label, col, marker, dx in COVERAGES:
        k = (rc.coverage == cov).to_numpy()
        d = pd.DataFrame({"nfl_id": rc.nfl_id[k], "b": pd.qcut(xw[k], 5, labels=False), "xw": xw[k], "yw": yw[k]})
        mid = d.groupby("b").xw.median()
        est = d.groupby("b").yw.mean()
        players = d.nfl_id.unique()
        rows = {p: np.flatnonzero(d.nfl_id.to_numpy() == p) for p in players}
        boots = []
        for _ in range(1000):
            idx = np.concatenate([rows[p] for p in rng.choice(players, len(players))])
            boots.append(d.iloc[idx].groupby("b").yw.mean().reindex(est.index).to_numpy())
        lo, hi = np.nanquantile(np.vstack(boots), [0.025, 0.975], axis=0)
        slope = np.polyfit(d.xw, d.yw, 1)[0]
        xs = np.array([mid.min(), mid.max()])
        ax.plot(xs, slope * xs, color=col, lw=1.2, ls="--", zorder=2)
        ax.vlines(mid + dx, lo, hi, color=col, lw=2, zorder=3)
        ax.plot(mid + dx, est, marker, ms=8, color=col, mec=SURFACE, mew=1.5, zorder=4, label=f"against {label}")
        if m == PRIMARY[0]:  # direct labels on the first panel; the legend carries the rest
            ax.annotate(f"against {label}", (mid.iloc[-1] + dx, est.iloc[-1]), xytext=(9, 0),
                        textcoords="offset points", va="center", fontsize=SMALL, color=INK)
        slopes.append(f"{label} {slope:+.2f}")
        note(f"F2 {NAMES[m]} against {label}: within-WR slope (yd per SD) / SOE gap, top vs bottom fifth (yd)",
             f"{slope:+.2f} / {est.iloc[-1] - est.iloc[0]:+.2f}")
    ax.axhline(0, color=INK2, lw=1, zorder=0)
    ax.set_title(f"{NAMES[m].capitalize()}\n{', '.join(slopes)} yd per SD", loc="left")
    ax.set_xlabel("vs the WR's own average (SD)")
axes[0].set_ylabel("separation over expected\nvs the WR's own average (yd)")
axes[-1].legend(frameon=False, fontsize=SMALL, loc="upper left", labelcolor=INK)
fig.suptitle("Within each receiver, a faster, sharper break buys separation against man coverage",
             x=0.01, ha="left", color=INK, fontsize=SUPTITLE)
fig.text(0.01, 0.905, f"{len(rc):,} routes with a coverage label, 62 WRs. Each WR's routes against one coverage, "
         "split into fifths by break score.\n95% player-bootstrap intervals; dashed lines = within-WR slopes.",
         ha="left", va="top", color=INK2, fontsize=SMALL)
fig.tight_layout(rect=(0, 0, 1, 0.83))
fig.savefig(FIGURES / "04_within.png", dpi=DPI)
plt.close(fig)

# %% [markdown]
# ## F4. The measurement ceiling: how high could the combine–game r have been?

# %%
crel = F.split_half_reliability(ccuts[ccuts.drill_type == "SKILL_DRILLS_WR"], n_iter=200,
                                metrics=tuple(PRIMARY)).set_index("feature").reliability
grel = split_half(gcuts, [f"{m}_z" for m in PRIMARY], players=q.index).set_index("feature").reliability
grel.index = grel.index.str.removesuffix("_z")
cn = ccuts[(ccuts.drill_type == "SKILL_DRILLS_WR") & ccuts.nfl_id.isin(q.index)].groupby("nfl_id").size().median()
gn = gcuts[gcuts.nfl_id.isin(q.index)].groupby("nfl_id").size().median()
note("median cuts per qualified WR: combine drills / games", f"{cn:.0f} / {gn:.0f}")
rows = []
for m in PRIMARY:
    r = q[f"pos_{m}"].corr(q[f"game_{m}"])
    z, se = np.arctanh(r), 1 / np.sqrt(len(q) - 3)
    lo, hi = np.tanh(z - 1.96 * se), np.tanh(z + 1.96 * se)
    ceil = np.sqrt(crel[m] * grel[m])
    rows.append({"m": m, "r": r, "lo": lo, "hi": hi, "ceiling": ceil,
                 "per_cut_combine": crel[m] / (cn - (cn - 1) * crel[m]),
                 "per_cut_game": grel[m] / (gn - (gn - 1) * grel[m])})
    note(f"F4 {NAMES[m]}: observed r [95% CI] / ceiling √(rel_c·rel_g)",
         f"{r:+.2f} [{lo:+.2f}, {hi:+.2f}] / {ceil:.2f}")
cdf = pd.DataFrame(rows).set_index("m")
for m in PRIMARY:
    note(f"F4 {NAMES[m]}: reliability of a single cut, combine / game",
         f"{cdf.loc[m, 'per_cut_combine']:.3f} / {cdf.loc[m, 'per_cut_game']:.3f}")
# Positive control (03b): top speed in the tracked 40 vs peak speed on go routes. Same Fisher-z interval as above.
pcr, pcn = positive_control["r"], positive_control["wrs"]
z, se = np.arctanh(pcr), 1 / np.sqrt(pcn - 3)
cdf.loc["top_speed", ["r", "lo", "hi", "ceiling"]] = [pcr, np.tanh(z - 1.96 * se), np.tanh(z + 1.96 * se),
                                                     positive_control["ceiling"]]
note("F4 positive control, top speed (tracked 40 → go routes): observed r [95% CI] / ceiling / WRs",
     f"{pcr:+.2f} [{cdf.loc['top_speed', 'lo']:+.2f}, {cdf.loc['top_speed', 'hi']:+.2f}] / "
     f"{positive_control['ceiling']:.2f} / {pcn}")
ROWS = [*PRIMARY, "top_speed"]
LABELS = {**NAMES, "top_speed": f"top speed: tracked 40 → go routes\n(positive control, {pcn} WRs)"}
fig, ax = plt.subplots(figsize=(9, 4.8))
ypos = np.array([4.0, 3.0, 2.0, 0.6])  # a gap sets the positive control apart
for y, m in zip(ypos, ROWS):
    rr = cdf.loc[m]
    ax.plot([0, rr.ceiling], [y, y], color=GRID, lw=8, solid_capstyle="butt", zorder=1)
    ax.plot(rr.ceiling, y, "|", ms=18, mew=2.5, color=INK2, zorder=2)
    ax.plot([rr.lo, rr.hi], [y, y], color=GAME, lw=2, zorder=3)
    ax.plot(rr.r, y, "o", ms=8, color=GAME, mec=SURFACE, mew=1.5, zorder=4)
ax.axhline(1.3, color=GRID, lw=1, ls=":", zorder=0)
ax.axvline(0, color=INK2, lw=1, zorder=0)
ax.set_yticks(ypos, [LABELS[m] for m in ROWS])
ax.tick_params(axis="y", length=0)
ax.grid(axis="y", visible=False)
ax.set_xlim(-0.35, 1.0)
ax.set_xlabel("correlation between the combine trait and the same trait in games")
legend = [Line2D([], [], color=GRID, lw=8, label="largest r measurement noise allows: √(rel. combine × rel. games)"),
          Line2D([], [], ls="", marker="o", ms=8, color=GAME, label="observed r, 95% interval")]
ax.legend(handles=legend, frameon=False, fontsize=SMALL, loc="upper center", bbox_to_anchor=(0.45, -0.2), ncol=1,
          labelcolor=INK)
fig.suptitle("The combine cut traits sit far below what measurement noise allows; top speed carries over",
             x=0.01, ha="left", color=INK, fontsize=SUPTITLE)
fig.tight_layout(rect=(0, 0, 1, 0.95))
fig.savefig(FIGURES / "04_ceiling.png", dpi=DPI)
plt.close(fig)

# %% [markdown]
# ## F6. How many routes before a WR's break metrics are trustworthy?

# %%
order = groutes[groutes.nfl_id.isin(q.index)].sort_values(["nfl_id", "season", "week", "game_id", "play_id"])
order["route_no"] = order.groupby("nfl_id").cumcount() + 1
kept = gcuts.merge(order[[*KEY, "route_no"]], on=KEY)
stab = []
for n in [25, 50, 100, 150, 200, 300, 400]:
    players = order.groupby("nfl_id").route_no.max().loc[lambda s: s >= n].index
    sub = kept[(kept.route_no <= n) & kept.nfl_id.isin(players)]
    rel = split_half(sub, [f"{m}_z" for m in PRIMARY], players=players, min_units=2)
    stab.append({"routes": n, "WRs": len(players), **{m: rel.set_index("feature").reliability[f"{m}_z"]
                                                     for m in PRIMARY},
                 "breaks": sub.groupby("nfl_id").size().median()})
stab = pd.DataFrame(stab)
print(stab.round(2).to_string())
fig, ax = plt.subplots(figsize=(8, 4.4))
# All three are game metrics, so none is orange (orange means "combine" in every figure).
for m, col, marker in zip(PRIMARY, [GAME, INK2, SERIES[2]], ["o", "^", "D"]):
    ax.plot(stab.routes, stab[m], "-", marker=marker, color=col, ms=7, mec=SURFACE, mew=1.2, label=NAMES[m])
ax.axhline(0.7, color=INK2, lw=1, ls="--")
ax.text(5, 0.71, "0.7", fontsize=SMALL, color=INK2, va="bottom", ha="left")
ax.axhline(0, color=INK2, lw=1, zorder=0)
ax.set_xlabel("a WR's first N regular-season routes")
ax.set_ylabel("split-half reliability")
ax.set_xlim(0, stab.routes.max() * 1.05)
ax.set_ylim(min(-0.3, stab[PRIMARY].min().min() - 0.05), 1)
ax.legend(frameon=False, fontsize=SMALL, loc="lower right", labelcolor=INK)
fig.suptitle("How fast in-game break metrics settle", x=0.01, ha="left", color=INK, fontsize=SUPTITLE)
fig.text(0.01, 0.885, "Each point uses the WRs with at least N routes (62 at N ≤ 100, 31 at N = 400)",
         ha="left", color=INK2, fontsize=SMALL)
fig.tight_layout(rect=(0, 0, 1, 0.88))
fig.savefig(FIGURES / "04_stabilize.png", dpi=DPI)
plt.close(fig)
for m in PRIMARY:
    hit = stab.loc[stab[m] >= 0.7, "routes"]
    note(f"F6 {NAMES[m]}: first route count with reliability ≥ 0.7",
         f"{int(hit.iloc[0])} routes" if len(hit) else f"not reached by {stab.routes.max()} (max {stab[m].max():.2f})")
for _, r in stab.iterrows():
    note(f"F6 reliability after {int(r.routes)} routes ({int(r.WRs)} WRs, median {r.breaks:.0f} breaks): "
         "entry / retention / lateral", f"{r.entry_speed:.2f} / {r.speed_retention:.2f} / {r.peak_lat_accel:.2f}")

# %% [markdown]
# ## F5. In-season scouting card (the F1 receiver)

# %%
def pct(col: str, v: float) -> float:
    return float((q[col] < v).mean() * 100)


card = []
for m in PRIMARY:
    card.append({"signal": NAMES[m], "combine percentile": pct(f"pos_{m}", wr.loc[pid, f"pos_{m}"]),
                 "game percentile": pct(f"game_{m}", wr.loc[pid, f"game_{m}"])})
card = pd.DataFrame(card)
soe_pct = pct("soe", wr.loc[pid, "soe"])
note(f"F5 {name}: combine vs game percentile among 62 qualified WRs",
     "; ".join(f"{r['signal']} {r['combine percentile']:.0f} vs {r['game percentile']:.0f}"
               for _, r in card.iterrows()))
note(f"F5 {name}: SOE percentile / routes / breaks", f"{soe_pct:.0f} / {int(wr.loc[pid, 'routes'])} / "
     f"{int(wr.loc[pid, 'game_n_cuts'])}")
# Robustness: the card uses the pre-registered combine features (every WR drill cut, including the Gauntlet).
# The matched cuts from 03b keep only each drill's designed route break; the combine percentiles should hold.
matched = pd.read_parquet(FEATURES / "combine_matched_features.parquet").reindex(q.index)
note(f"F5 {name}: combine percentile on matched cuts (03b), for comparison",
     "; ".join(f"{NAMES[m]} {(matched[f'match_{m}'] < matched.loc[pid, f'match_{m}']).mean() * 100:.0f}"
               for m in PRIMARY) + f" ({int(matched.loc[pid, 'match_n_cuts'])} cuts)")
fig, ax = plt.subplots(figsize=(8, 3.6))
ypos = np.arange(len(card))[::-1]
for y, (_, r) in zip(ypos, card.iterrows()):
    ax.plot([0, 100], [y, y], color=GRID, lw=6, solid_capstyle="round", zorder=1)
    ax.plot([r["combine percentile"], r["game percentile"]], [y, y], color=INK2, lw=1.5, zorder=2)
    ax.plot(r["combine percentile"], y, "s", ms=9, color=COMBINE, mec=SURFACE, mew=1.5, zorder=3)
    ax.plot(r["game percentile"], y, "o", ms=9, color=GAME, mec=SURFACE, mew=1.5, zorder=3)
ax.set_yticks(ypos, card.signal)
ax.tick_params(axis="y", length=0)
ax.grid(axis="y", visible=False)
ax.set_xlim(-2, 102)
ax.set_xlabel("percentile among the 62 qualified WRs")
legend = [Line2D([], [], ls="", marker="s", ms=9, color=COMBINE, label="combine drills (draft day)"),
          Line2D([], [], ls="", marker="o", ms=9, color=GAME,
                 label=f"games ({int(wr.loc[pid, 'game_n_cuts'])} breaks)")]
ax.legend(handles=legend, frameon=False, fontsize=SMALL, loc="upper center", bbox_to_anchor=(0.5, -0.3), ncol=2,
          labelcolor=INK)
ax.set_title(f"{name} · separation over expected: {soe_pct:.0f}th percentile", loc="left")
fig.tight_layout()
fig.savefig(FIGURES / "04_card.png", dpi=DPI)
plt.close(fig)

# %% [markdown]
# ## Context numbers for the prose

# %%
note("WRs in the cohort / with combine WR-drill features / qualified",
     f"{len(wr)} / {wr.pos_entry_speed.notna().sum()} / {len(q)}")
n_windows = len(groutes[groutes.t_pass.notna() & groutes.t_snap.notna()])
note("game route windows / route breaks kept", f"{n_windows:,} / {len(gcuts):,}")
note("route-level SOE SD (Test 2 routes, yd)", f"{rt.soe.std():.2f}")
note("player-level SOE SD (qualified WRs, yd)", f"{q.soe.std():.2f}")

numbers[4:4] = ["## Checks", "", *checks, ""]
(ROOT / "reports" / "04_numbers.md").write_text("\n".join(numbers) + "\n", encoding="utf-8")
print(f"\n{len(checks)} checks passed; wrote reports/04_numbers.md")

## Run the six steps

In [ ]:
def run(step: str) -> None:
    r = subprocess.run([sys.executable, f"notebooks/{step}.py"], capture_output=True, text=True)
    (PROJECT / "logs" / f"{step}.log").write_text(r.stdout + r.stderr)
    if r.returncode != 0:
        raise RuntimeError(f"{step} failed:\n{r.stderr[-4000:]}")
    print(f"{step}: {r.stdout.strip().splitlines()[-1]}")


for step in ['00_audit', '01_combine_features', '02_game_features', '03_models', '03b_sensitivity', '04_writeup']:
    run(step)

## Results

The pre-registered tests from `reports/03_models.md`, the checks added after them from `reports/03b_sensitivity.md`,
then the writeup's figures. The full reports for every step are in `bdb/reports/`.

In [ ]:
from IPython.display import Image, Markdown, display

report = (PROJECT / "reports" / "03_models.md").read_text()
start = report.index("## Test 1")
end = report.index("## Secondary analyses")
display(Markdown(report[start:end]))

### Added after the pre-registered tests (not pre-registered)

In [ ]:
checks = (PROJECT / "reports" / "03b_sensitivity.md").read_text()
display(Markdown(checks[checks.index("## Tests 1 and 3 on matched combine cuts"):]))

*Figure 1. One receiver's combine speed-out vs the same receiver's game out-breaks.*

In [ ]:
display(Image(filename=str(PROJECT / "reports" / "figures" / "04_signature.png")))

*Figure 2. The three pre-registered tests.*

In [ ]:
display(Image(filename=str(PROJECT / "reports" / "figures" / "03_tests.png")))

*Figure 3. Within each receiver, separation rises with the break against man coverage.*

In [ ]:
display(Image(filename=str(PROJECT / "reports" / "figures" / "04_within.png")))

*Figure 4. The measurement ceiling vs the observed combine–game correlation, with the positive control.*

In [ ]:
display(Image(filename=str(PROJECT / "reports" / "figures" / "04_ceiling.png")))

*Figure 5. In-season scouting card.*

In [ ]:
display(Image(filename=str(PROJECT / "reports" / "figures" / "04_card.png")))

*Figure 6. How fast in-game break metrics settle.*

In [ ]:
display(Image(filename=str(PROJECT / "reports" / "figures" / "04_stabilize.png")))